# 📊 26 · Categorical data and multiple testing
### The χ² goodness-of-fit test on Mendel's peas, the χ² distribution and its table, Hardy–Weinberg tests, contingency tables and Fisher's exact test — then thousands of tests at once: the family-wise error rate, Bonferroni, the false discovery rate and Benjamini–Hochberg, a 400-gene differential-expression screen and gene-set enrichment

**BiologyToMath** · Part G — Statistical inference in practice · Module 26 of 28

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/math/tutorial/BiologyToMath/notebooks/26_categorical_tests_multiple_testing.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`scipy.stats` — to check hand-written formulas), pandas, Matplotlib |
| **Prerequisites** | [Module 25 · 🚦 Hypothesis testing: errors, power, z- and t-tests](25_hypothesis_testing.ipynb) (tests, $p$-values, Type I errors, Welch's $t$-test); [Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb) (binomial coefficients, the binomial distribution); [Module 17 · 🔀 Conditional probability, independence and Bayes' rule](17_conditional_probability_bayes.ipynb) (independence); [Module 23 · 🎯 Parameter estimation and maximum likelihood](23_maximum_likelihood.ipynb) (maximum-likelihood allele frequency); [Module 03 · 🧺 Sets and functions: collections and mappings](03_sets_and_functions.ipynb) (sets and Venn diagrams). |
| **Estimated time** | 170 min (reading + running + the ✏️ exercises) |
| **02-680 topics covered** | Testing categorical data: hypothesised category probabilities $\dot p$ vs observed proportions $p$, $H_0:\dot p = p$; Mendel's dihybrid cross (9:3:3:1); the χ² goodness-of-fit statistic and its degrees of freedom; the χ² distribution (density, sum of squared standard normals) and χ² lookup tables; Hardy–Weinberg tests; χ² tests of independence for contingency tables; Fisher's exact test; multiple testing: expected false positives, family-wise error rate, Bonferroni correction, false discovery rate and the Benjamini–Hochberg procedure; enrichment tests |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. Mendel's pea counts (315, 101, 108, 32) are **typed in** from his 1866 paper. **All other data are SIMULATED by code**: Hardy–Weinberg and case–control genotypes (`simulate_genotypes`), $p$-values of simulated screens, and the 400-gene infection experiment (`simulate_tissue_expression`), whose truly changed genes are known. The rare-variant table of Section 5 is illustrative. The χ² table is generated with `scipy.stats.chi2`.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **State** a goodness-of-fit hypothesis $H_0:\ \dot p = p$ for categorical data and **compute** expected counts from
   a genetic model (Mendel's 9:3:3:1).
2. **Compute** the χ² statistic $\sum_i (O_i-E_i)^2/E_i$ by hand and in code, **choose** its degrees of freedom, and
   **explain** why it is approximately χ²-distributed (and **prove** it for two categories).
3. **Write down** the χ² density, **verify** that a sum of $p$ squared standard normals is $\chi^2_p$, and **read** a χ²
   table you generated yourself — in both tails ("too good to be true?").
4. **Test** Hardy–Weinberg equilibrium with an estimated allele frequency and **explain** why it costs a degree of freedom.
5. **Test** independence in an $r\times c$ contingency table (genotype × disease), **derive** the expected counts, and
   **use** Fisher's exact test when counts are small.
6. **Derive** the expected number of false positives $m\alpha$ and the family-wise error rate $1-(1-\alpha)^m$, and
   **prove** that Bonferroni controls the FWER.
7. **Implement** the Benjamini–Hochberg procedure and **demonstrate** by simulation that it controls the false
   discovery rate.
8. **Run** a complete differential-expression screen on 400 genes in three tissues, **evaluate** it against the known
   truth (precision, recall, Venn diagrams) and **test** a gene set for enrichment with the hypergeometric distribution.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, Mendel's counts | 5 min |
| 1 · Testing categorical data: Mendel's peas | Punnett square in code, $H_0:\dot p = p$, expected counts | 15 min |
| 2 · The χ² goodness-of-fit statistic | compute it, degrees of freedom, why it is χ² (proof for $k=2$, simulation) | 20 min |
| 3 · The χ² distribution | density, sums of squared normals, the χ² table, "too good to be true?" | 20 min |
| 4 · Hardy–Weinberg equilibrium | a test with an estimated parameter; inbreeding | 15 min |
| 5 · Contingency tables | genotype × disease, expected = row × column / n, allelic test, Fisher's exact test | 20 min |
| 6 · The multiple-testing problem | $m\alpha$ false positives, FWER, Bonferroni | 15 min |
| 7 · The false discovery rate | Benjamini–Hochberg by hand and in code; simulation | 20 min |
| 8 · A genome-wide screen | 400 genes × 3 tissues: $p$-value histograms, volcano plots, truth tables, Venn diagrams | 25 min |
| 9 · Gene-set enrichment | the hypergeometric test for a pathway | 10 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 10 min |

## 🧫 The biological question

In 1866 Gregor Mendel reported the result of crossing pea plants that were heterozygous for two traits — seed shape
(round, dominant, vs wrinkled) and seed colour (yellow, dominant, vs green). His theory of independently assorting
factors predicted the four seed types in the ratio **9 : 3 : 3 : 1**. He counted **556 seeds: 315 round-yellow,
101 wrinkled-yellow, 108 round-green, 32 wrinkled-green.** ***Do these counts agree with the theory?*** The numbers are
not exactly $\frac{9}{16}\cdot556 = 312.75$, $104.25$, $104.25$, $34.75$ — but how far off may random counts be before we
doubt the theory?

The same question — *observed category counts vs a model* — is asked in every genetics lab: are genotype frequencies in
Hardy–Weinberg proportions? Is a risk allele more common in patients than in controls? And modern genomics adds a twist
Mendel never faced: a single experiment now runs **thousands of tests** — one per gene or per SNP — and at α = 0.05
hundreds of them come out "significant" by chance alone.

| Biological question | Mathematical tool | Where |
|---|---|---|
| Do Mendel's counts fit 9:3:3:1? | χ² goodness-of-fit test | Sections 1–3 |
| Are genotypes in Hardy–Weinberg proportions? | χ² test with an estimated parameter (1 df) | Section 4 |
| Is a genotype associated with a disease? | χ² test of independence; Fisher's exact test | Section 5 |
| How many false positives in 400 tests? | $m\alpha$, FWER, Bonferroni | Section 6 |
| Which genes changed after infection? | false discovery rate, Benjamini–Hochberg | Sections 7–8 |
| Is the interferon pathway over-represented? | hypergeometric enrichment test | Section 9 |

> 🧪 **Simulated data, with a known truth.** Apart from Mendel's counts, every data set is simulated, so for each test we
> know whether $H_0$ was really true — and can count false discoveries exactly.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import itertools
from math import comb, exp, lgamma

from scipy import stats

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py). `simulate_genotypes` draws SIMULATED 0/1/2 genotypes (Hardy–Weinberg, optionally with inbreeding); `simulate_tissue_expression` is the course's SIMULATED infection experiment (400 genes, 3 tissues, known truly-changed genes); `draw_venn` and `jaccard` compare gene sets as in Module 03. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def jaccard(a, b) -> float:
    """Jaccard index |A ∩ B| / |A ∪ B| of two collections (1.0 for two empty sets)."""
    a, b = set(a), set(b)
    union = a | b
    return len(a & b) / len(union) if union else 1.0


def _venn_order(x):
    """Sort key for Venn-diagram labels: numbers in numeric order first, then everything else as text."""
    if isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, bool):
        return (0, float(x), "")
    return (1, 0.0, str(x))


def draw_venn(sets: dict, universe=None, ax=None, show_elements: bool = True, max_items: int = 6,
              title: str | None = None, highlight=None):
    """Draw a Venn diagram of two or three sets with matplotlib only.

    ``sets`` maps a label to a Python set, e.g. ``{"S": {1, 2, 3}, "T": {3, 4}}``.  Each region
    shows its elements (or ``n = …`` if it has more than ``max_items``).  If ``universe`` is given,
    a rectangle is drawn and the elements outside every set are listed in a corner.
    ``highlight`` is a function of the membership tuple, e.g. ``lambda s, t: s and not t`` for
    S \\ T; the regions where it is true are shaded.  Returns the Axes.
    """
    labels = list(sets)
    if len(labels) not in (2, 3):
        raise ValueError("draw_venn draws two or three sets")
    if ax is None:
        _, ax = plt.subplots(figsize=(4.6, 3.8))
    if len(labels) == 2:
        centres = [(-0.55, 0.0), (0.55, 0.0)]
    else:
        centres = [(-0.55, 0.32), (0.55, 0.32), (0.0, -0.6)]
    radius = 1.0
    colours = ["tab:blue", "tab:orange", "tab:green"]
    # classify a fine grid of points to find each region (used for shading and label positions)
    gx, gy = np.meshgrid(np.linspace(-2.2, 2.2, 441), np.linspace(-2.0, 1.9, 391))
    inside = [((gx - cx) ** 2 + (gy - cy) ** 2) <= radius ** 2 for cx, cy in centres]
    if highlight is not None:
        mask = np.zeros_like(gx, dtype=bool)
        for member in itertools.product([True, False], repeat=len(labels)):
            if any(member) and highlight(*member):
                region = np.ones_like(gx, dtype=bool)
                for k, m in enumerate(member):
                    region &= inside[k] if m else ~inside[k]
                mask |= region
        ax.contourf(gx, gy, mask.astype(float), levels=[0.5, 1.5], colors=["#ffd54f"], alpha=0.8)
    for (cx, cy), lab, col in zip(centres, labels, colours):
        ax.add_patch(plt.Circle((cx, cy), radius, fill=False, lw=2, color=col))
        lx = cx + (-1.0 if cx < 0 else 1.0 if cx > 0 else 0.0) * 0.95
        ly = cy + (0.95 if cy >= 0 else -1.15)
        ax.text(lx, ly, lab, ha="center", va="center", fontsize=11, fontweight="bold", color=col)
    elements = set().union(*sets.values())
    for member in itertools.product([True, False], repeat=len(labels)):
        if not any(member):
            continue
        region_items = [x for x in elements if all((x in sets[l]) == m for l, m in zip(labels, member))]
        region = np.ones_like(gx, dtype=bool)
        for k, m in enumerate(member):
            region &= inside[k] if m else ~inside[k]
        if not region.any():
            continue
        px, py = gx[region].mean(), gy[region].mean()
        listing = ", ".join(str(x) for x in sorted(region_items, key=_venn_order))
        if show_elements and len(region_items) <= max_items and len(listing) <= 16:
            text = listing if region_items else "∅"
        else:
            text = f"n = {len(region_items)}"
        ax.text(px, py, text, ha="center", va="center", fontsize=8, wrap=True)
    if universe is not None:
        ax.add_patch(plt.Rectangle((-2.1, -1.9), 4.2, 3.7, fill=False, lw=1, color="0.4"))
        outside = sorted(set(universe) - elements, key=_venn_order)
        text = ", ".join(map(str, outside)) if len(outside) <= max_items else f"n = {len(outside)}"
        ax.text(-2.0, -1.8, "U: " + (text if outside else "∅"), fontsize=8, va="bottom", color="0.3")
    ax.set_xlim(-2.2, 2.2)
    ax.set_ylim(-2.0, 1.9)
    ax.set_aspect("equal")
    ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


# Named genes of the simulated infection experiment.  The infection-response genes are real genes
# known to be induced during infection (cytokines, interferon-stimulated genes, acute-phase
# proteins; lysozyme, LYZ, is an antimicrobial enzyme rather than a classic acute-phase protein and is
# grouped with them only for simplicity); the reference genes are classic "housekeeping" genes used
# to normalise qPCR.
INFECTION_PATHWAYS = {
    "cytokine signalling": ["IL1B", "TNF", "IL6", "CXCL8"],
    "interferon response": ["IRF1", "STAT1", "MX1", "ISG15"],
    "acute phase": ["SAA1", "HAMP", "C3", "LYZ"],
}


HOUSEKEEPING_GENES = ["ACTB", "GAPDH", "EEF1A1", "RPL13A", "TBP", "HPRT1",
                      "PPIA", "UBC", "YWHAZ", "SDHA", "TUBA1A", "B2M"]


def simulate_tissue_expression(n_genes: int = 400, tissues=("spleen", "kidney", "liver"), n_reps: int = 4,
                               seed: int = 2680) -> dict:
    """SIMULATED log2 gene expression from an infection experiment in several tissues.

    Every tissue has ``n_reps`` control and ``n_reps`` infected samples.  The 12 genes of
    ``INFECTION_PATHWAYS`` respond to infection in every tissue (up by 1.5–3 log2 units); further
    "g…" genes respond in one tissue only or in two tissues, so the sets of truly changed genes
    overlap like the differentially-expressed-gene sets of a real multi-tissue study.  The 12
    ``HOUSEKEEPING_GENES`` are stable (tiny noise, no response).  About a third of all genes differ in
    baseline level between tissues.

    Returns a dict with ``X`` (samples × genes, log2 expression), ``samples`` (DataFrame with
    columns sample, tissue, condition), ``genes`` (list), ``true_lfc`` (DataFrame tissues × genes,
    true infected − control difference), ``true_de`` (dict tissue → set of truly changed genes),
    ``pathways`` (dict pathway → genes) and ``source``.
    """
    rng = np.random.default_rng(seed)
    named = [g for genes in INFECTION_PATHWAYS.values() for g in genes]
    n_other = n_genes - len(named) - len(HOUSEKEEPING_GENES)
    if n_other < 60:
        raise ValueError("n_genes must be at least 84")
    genes = named + HOUSEKEEPING_GENES + [f"g{i:03d}" for i in range(1, n_other + 1)]
    J, T = len(genes), len(tissues)
    base = rng.normal(7.0, 1.8, J)
    base[len(named):len(named) + len(HOUSEKEEPING_GENES)] = rng.uniform(9.0, 12.0, len(HOUSEKEEPING_GENES))
    tissue_shift = rng.normal(0.0, 1.0, (T, J)) * (rng.random(J) < 0.35)
    tissue_shift[:, len(named):len(named) + len(HOUSEKEEPING_GENES)] = 0.0
    lfc = np.zeros((T, J))
    lfc[:, :len(named)] = rng.uniform(1.5, 3.0, (T, len(named)))
    other = list(range(len(named) + len(HOUSEKEEPING_GENES), J))
    rng.shuffle(other)
    sizes_single = [24, 18, 14, 12, 10][:T]
    pairs = list(itertools.combinations(range(T), 2))
    sizes_pair = [10, 5, 4, 3, 3, 2, 2, 2, 2, 2][:len(pairs)]
    k = 0

    def effect(m):
        return rng.uniform(1.0, 2.5, m) * np.where(rng.random(m) < 0.7, 1.0, -1.0)

    for t, m in enumerate(sizes_single):
        idx = other[k:k + m]
        lfc[t, idx] = effect(m)
        k += m
    for (t1, t2), m in zip(pairs, sizes_pair):
        idx = other[k:k + m]
        e = effect(m)
        lfc[t1, idx] = e
        lfc[t2, idx] = e * rng.uniform(0.8, 1.2, m)
        k += m
    noise_sd = rng.uniform(0.25, 0.55, J)
    noise_sd[len(named):len(named) + len(HOUSEKEEPING_GENES)] = 0.08
    rows, X = [], []
    for t, tissue in enumerate(tissues):
        for condition in ("control", "infected"):
            for r in range(1, n_reps + 1):
                mean = base + tissue_shift[t] + (lfc[t] if condition == "infected" else 0.0)
                X.append(mean + rng.normal(0.0, noise_sd))
                rows.append((f"{tissue}_{condition[:3]}{r}", tissue, condition))
    samples = pd.DataFrame(rows, columns=["sample", "tissue", "condition"])
    true_lfc = pd.DataFrame(lfc, index=list(tissues), columns=genes)
    true_de = {tissue: {g for g, v in zip(genes, lfc[t]) if v != 0} for t, tissue in enumerate(tissues)}
    return {"X": np.array(X), "samples": samples, "genes": genes, "true_lfc": true_lfc,
            "true_de": true_de, "pathways": {k_: list(v) for k_, v in INFECTION_PATHWAYS.items()},
            "source": "SIMULATED log2 expression (BiologyToMath simulate_tissue_expression)"}


def simulate_genotypes(n: int, p: float, inbreeding: float = 0.0, seed: int = 1866) -> np.ndarray:
    """SIMULATED genotypes of ``n`` diploid individuals at one SNP, coded 0/1/2 = number of copies of
    the allele with frequency ``p``.  With ``inbreeding`` F = 0 the genotype frequencies follow
    Hardy–Weinberg proportions ((1−p)², 2p(1−p), p²); F > 0 moves individuals from heterozygotes to
    homozygotes: P(1) = 2p(1−p)(1−F)."""
    rng = np.random.default_rng(seed)
    q = 1.0 - p
    probs = np.array([q * q + inbreeding * p * q, 2 * p * q * (1 - inbreeding), p * p + inbreeding * p * q])
    return rng.choice(3, size=n, p=probs / probs.sum())

print("🧰 toolkit loaded:", ", ".join(sorted(['HOUSEKEEPING_GENES', 'INFECTION_PATHWAYS', '_venn_order', 'draw_venn', 'jaccard', 'simulate_genotypes', 'simulate_tissue_expression'])))

In [ ]:
# ── 0.1 Mendel's dihybrid cross (typed in from Mendel 1866) ──
categories = ["round/yellow", "wrinkled/yellow", "round/green", "wrinkled/green"]
observed = np.array([315, 101, 108, 32])                 # Mendel's counts of F2 seeds
p_dot = np.array([9, 3, 3, 1]) / 16                       # hypothesised category probabilities (ṗ)
n_seeds = observed.sum()
expected = n_seeds * p_dot                                # expected counts n·ṗ_i
mendel = pd.DataFrame({"observed O": observed, "ṗ (theory)": p_dot, "expected E = n·ṗ": expected,
                       "observed proportion p": observed / n_seeds}, index=categories)
print(f"n = {n_seeds} seeds")
print(mendel.round(4).to_string())

🔍 **What just happened?** We typed in Mendel's four counts (total 556) and the probabilities of his theory,
$\dot p = (\tfrac{9}{16}, \tfrac{3}{16}, \tfrac{3}{16}, \tfrac{1}{16})$. Multiplying by $n = 556$ gives the expected counts
**312.75, 104.25, 104.25, 34.75**. The observed proportions (0.5665, 0.1817, 0.1942, 0.0576) are close to
0.5625, 0.1875, 0.1875, 0.0625 — but not equal. Is the gap bigger than chance would produce?

---
## 1 · Testing categorical data: Mendel's peas

> 📍 **Where we are.** In [Module 25 · 🚦 Hypothesis testing: errors, power, z- and t-tests](25_hypothesis_testing.ipynb) the data were numbers (cholesterol levels) and the hypothesis was about a mean. Here the
> data are **categories** — each seed is one of four types — and the hypothesis is about the **probabilities** of the
> categories. The recipe is the same: statistic → $p$-value → decision.

**Where 9:3:3:1 comes from.** Write $R$/$r$ for the round/wrinkled alleles and $Y$/$y$ for yellow/green; $R$ and $Y$ are
dominant. A doubly heterozygous plant $RrYy$ makes four kinds of gametes — $RY$, $Ry$, $rY$, $ry$ — each with probability
$\tfrac14$ if the two genes assort independently (they are on different chromosomes, or far apart on one). Crossing two
such plants pairs a random gamete from each parent: a $4\times4$ Punnett square, i.e. the Cartesian product of the two
gamete sets (Modules 04 and 16), with 16 equally likely cells.

In [ ]:
# ── 1.1 The Punnett square of RrYy × RrYy, enumerated (Cartesian product of gametes) ──
gametes = ["RY", "Ry", "rY", "ry"]


def phenotype(g1, g2):
    """Seed phenotype of the zygote formed by two gametes: round if at least one R, yellow if at least one Y."""
    shape = "round" if "R" in (g1[0], g2[0]) else "wrinkled"
    colour = "yellow" if "Y" in (g1[1], g2[1]) else "green"
    return f"{shape}/{colour}"


square = pd.DataFrame([[phenotype(a, b) for b in gametes] for a in gametes], index=gametes, columns=gametes)
print("Punnett square (rows: gamete from parent 1, columns: gamete from parent 2):")
print(square.to_string())
counts16 = pd.Series([phenotype(a, b) for a, b in itertools.product(gametes, gametes)]).value_counts()
print("\ncells of each phenotype out of 16:", counts16[categories].to_dict())

🔍 **What just happened?** Enumerating the 16 gamete pairs reproduces Mendel's prediction: **9** cells are round/yellow,
**3** wrinkled/yellow, **3** round/green and **1** wrinkled/green. So under the theory each seed independently falls in
the four categories with probabilities $\dot p = (9, 3, 3, 1)/16$.

**The hypotheses.** Suppose a theory assigns probabilities $\dot p = (\dot p_1,\dots,\dot p_k)$ to $k$ categories, and we
observe counts $O_1,\dots,O_k$ in $n$ independent trials, which we convert to observed proportions
$p = (p_1,\dots,p_k)$ with $p_i = O_i/n$ (estimates of the true category probabilities). We test

$$H_0:\ \dot p = p \qquad\text{vs}\qquad H_1:\ \dot p \ne p, \qquad\text{where}\quad \sum_{i=1}^k\dot p_i = 1\ \text{ and }\ \sum_{i=1}^k p_i = 1 .$$

*In words:* $H_0$ says the true category probabilities are those of the theory (any difference between the observed
proportions and $\dot p$ is sampling noise); $H_1$ says at least one category probability differs. Both are probability
vectors, so each sums to 1. (Strictly, the hypothesis is about the *true* proportions, of which $p$ is our estimate.)

In [ ]:
# ── 1.2 Observed vs expected counts ──
fig, ax = plt.subplots(figsize=(6.4, 3.4))
xpos = np.arange(4)
ax.bar(xpos - 0.2, observed, width=0.4, color="tab:green", label="observed (Mendel)")
ax.bar(xpos + 0.2, expected, width=0.4, color="0.7", label="expected under 9:3:3:1")
for x, o, e in zip(xpos, observed, expected):
    ax.text(x - 0.2, o + 5, f"{o}", ha="center", fontsize=8)
    ax.text(x + 0.2, e + 5, f"{e:.2f}", ha="center", fontsize=8)
ax.set_xticks(xpos); ax.set_xticklabels(categories)
ax.set_ylabel("number of seeds"); ax.legend()
ax.set_title("Mendel's dihybrid cross: 556 seeds")
plt.tight_layout(); plt.show()
print("O − E:", observed - expected)

🔍 **What just happened?** The bars are almost indistinguishable: the differences $O-E$ are **+2.25, −3.25, +3.75,
−2.75** seeds. Note that they sum to zero — they must, because both the observed and the expected counts add up to 556.
We need a single number that summarises how far the four counts are from the theory, and a distribution for that number
under $H_0$.

✏️ **Your turn.** A monohybrid cross $Aa\times Aa$ predicts dominant : recessive = 3 : 1. Mendel counted 5,474 round and
1,850 wrinkled seeds. Compute the expected counts and the differences $O-E$ (you will test them in Section 2).

---
## 2 · The χ² goodness-of-fit statistic

> 📍 **Where we are.** We have observed and expected counts. Karl Pearson's idea (1900): square each difference, scale it
> by what is expected, and add up.

$$X^2 \;=\; \sum_{i=1}^{k}\frac{(O_i-E_i)^2}{E_i} \;=\; \sum_{i=1}^{k}\frac{(c_i-n\,\dot p_i)^2}{n\,\dot p_i}\ \ \overset{\text{approx.}}{\sim}\ \ \chi^2_{k-1}\quad\text{under }H_0 .$$

*In words:* for each category, take the difference between the count $c_i = O_i$ and the expected count $E_i = n\dot p_i$,
square it (so that positive and negative deviations both count), divide by the expected count (a deviation of 3 matters
more when 35 are expected than when 313 are), and add over the $k$ categories. Under $H_0$ the result approximately follows
a χ² distribution with $k-1$ degrees of freedom (Section 3). Large values = poor fit.

⚠️ **Common confusion — the denominator is the expected *count* $n\dot p_i$, not the probability $\dot p_i$.** Dividing by
$\dot p_i$ would multiply the statistic by $n$ and make every large study "significant". We check this in the next cell.

**Degrees of freedom.** There are $k$ counts, but they must add up to $n$: once $k-1$ of them are known, the last is
fixed. So the statistic has $k-1$ free pieces of information — for Mendel's four categories, **3 degrees of freedom**.
(Same idea as $n-1$ for the sample variance in [Module 25 · 🚦 Hypothesis testing: errors, power, z- and t-tests](25_hypothesis_testing.ipynb).)

In [ ]:
# ── 2.1 The statistic by hand, term by term, and with SciPy ──
terms = (observed - expected) ** 2 / expected
x2 = terms.sum()
for cat, o, e, t in zip(categories, observed, expected, terms):
    print(f"{cat:16s} ({o:3d} − {e:6.2f})² / {e:6.2f} = {t:.4f}")
print(f"X² = {x2:.4f}   with k − 1 = {len(observed) - 1} degrees of freedom")
p_mendel = stats.chi2.sf(x2, df=3)
print(f"p-value = P(χ²_3 ≥ {x2:.3f}) = {p_mendel:.4f}")
print("scipy.stats.chisquare:", stats.chisquare(observed, expected))
print(f"\n(wrong denominator ṗ_i instead of n·ṗ_i: {((observed - expected) ** 2 / p_dot).sum():.1f} "
      f"= n × X² = {n_seeds * x2:.1f} → p = {stats.chi2.sf(((observed - expected) ** 2 / p_dot).sum(), 3):.1e})")

🔍 **What just happened?** The four terms are 0.0162, 0.1013, 0.1349 and 0.2176; their sum is **X² = 0.470**. With 3
degrees of freedom the $p$-value is **0.925**: if Mendel's theory is right, 92.5 % of repetitions of his experiment would
fit *worse* than his did. There is no evidence against 9:3:3:1 — **we retain $H_0$**. `scipy.stats.chisquare` agrees.
With the wrong denominator the "statistic" would be 261.3 — 556 times larger — and the $p$-value about $10^{-56}$: a
perfectly good fit would be "rejected" with absurd confidence.

### Why is $X^2$ approximately χ²? A proof for two categories

Take $k=2$ categories with probabilities $\dot p$ and $1-\dot p$. Then $O_1\sim\text{Binomial}(n,\dot p)$ (Module 18), and
$O_2 = n-O_1$, $E_1 = n\dot p$, $E_2 = n(1-\dot p)$. Since the counts sum to $n$, $O_2-E_2 = -(O_1-E_1)$. So

$$\begin{aligned}
X^2 &= \frac{(O_1-E_1)^2}{E_1}+\frac{(O_2-E_2)^2}{E_2} = (O_1-E_1)^2\Big(\frac{1}{n\dot p}+\frac{1}{n(1-\dot p)}\Big)\\
&= (O_1-E_1)^2\,\frac{(1-\dot p)+\dot p}{n\dot p(1-\dot p)} = \left(\frac{O_1-n\dot p}{\sqrt{n\dot p(1-\dot p)}}\right)^2 = Z^2 .
\end{aligned}$$

*In words:* with two categories, $X^2$ is exactly the square of the standardised binomial count $Z$ (subtract the mean
$n\dot p$, divide by the standard deviation $\sqrt{n\dot p(1-\dot p)}$). By the central limit theorem (Module 22) $Z$ is
approximately $\mathcal N(0,1)$, so $X^2$ is approximately the square of a standard normal — which is the definition of
$\chi^2_1$ (Section 3). With $k$ categories the same argument (Pearson 1900) gives a sum of $k-1$ independent squared
standard normals, i.e. $\chi^2_{k-1}$: one dimension is lost to the constraint $\sum O_i = n$.

Let us check the approximation for Mendel's design by simulating 20,000 experiments under $H_0$.

In [ ]:
# ── 2.2 The null distribution of X² by simulation: 20,000 simulated repeats of the Mendel experiment under 9:3:3:1 ──
rng_mult = np.random.default_rng(SEED + 22)
sim_counts = rng_mult.multinomial(n_seeds, p_dot, size=20_000)       # each row: 4 counts summing to 556
sim_x2 = ((sim_counts - expected) ** 2 / expected).sum(axis=1)

grid = np.linspace(0.01, 16, 400)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.hist(sim_x2, bins=40, range=(0, 16), density=True, color="0.75", label="X² of 20,000 simulated experiments")
ax.plot(grid, stats.chi2.pdf(grid, 3), color="tab:red", label="χ² density, 3 degrees of freedom")
ax.axvline(x2, color="tab:green", lw=2, label=f"Mendel's X² = {x2:.2f}")
ax.set_xlabel("X²"); ax.set_ylabel("density"); ax.legend()
ax.set_title("null distribution of the goodness-of-fit statistic")
plt.tight_layout(); plt.show()
print(f"simulated P(X² ≥ {x2:.2f}) = {np.mean(sim_x2 >= x2):.4f}   χ²_3 tail = {p_mendel:.4f}")
print(f"simulated 95th percentile = {np.quantile(sim_x2, 0.95):.3f}   χ²_3 95th percentile = {stats.chi2.ppf(0.95, 3):.3f}")

🔍 **What just happened?** The histogram of $X^2$ from 20,000 simulated "Mendel experiments" (multinomial counts with
probabilities 9:3:3:1, $n = 556$) follows the $\chi^2_3$ curve closely. The simulated tail probability beyond 0.47
matches the χ² $p$-value of 0.925, and the simulated 95th percentile is within a few hundredths of the theoretical cut-off
7.815. The approximation is good when every expected count is reasonably large — a common rule of thumb asks for $E_i\ge5$
in every category (Mendel's smallest is 34.75).

### 📘 Lecture notes — the χ² goodness-of-fit test

- **Setting:** $k$ categories with hypothesised probabilities $\dot p = (\dot p_1,\dots,\dot p_k)$; observed counts
  $c_1,\dots,c_k$ in $n$ trials, i.e. observed proportions $p = (p_1,\dots,p_k)$. $H_0:\dot p = p$ vs $H_1:\dot p\ne p$, with
  $\sum\dot p_i = 1$ and $\sum p_i = 1$.
- **Statistic:** $X^2 = \sum_{i=1}^k\frac{(c_i-n\dot p_i)^2}{n\dot p_i} = \sum\frac{(O-E)^2}{E}\ \sim\ \chi^2_{k-1}$
  approximately under $H_0$ (denominator = expected **count**).
- **Degrees of freedom:** number of categories − 1 (the counts must sum to $n$).
- **Mendel's dihybrid cross:** $\dot p = (\frac9{16},\frac3{16},\frac3{16},\frac1{16})$ for round/yellow, wrinkled/yellow,
  round/green, wrinkled/green; observed 315, 101, 108, 32 ($n=556$); expected 312.75, 104.25, 104.25, 34.75;
  $X^2 = \frac{(315-312.75)^2}{312.75}+\frac{(101-104.25)^2}{104.25}+\frac{(108-104.25)^2}{104.25}+\frac{(32-34.75)^2}{34.75}\approx0.47$,
  3 df, **$p\approx0.925$** → we cannot reject $H_0$.
- **Why χ²:** for $k=2$, $X^2 = Z^2$ with $Z$ the standardised binomial count; in general $X^2$ ≈ a sum of $k-1$ squared
  standard normals. Needs large expected counts (rule of thumb $E_i\ge5$).

> 🎓 **Exam tip.** Write the statistic with the expected **counts** in the denominator, compute df as categories − 1 (minus one more for every parameter estimated from the data, Section 4), and remember that only *large* $X^2$ is evidence against $H_0$.

✏️ **Your turn.** Finish the monohybrid exercise of Section 1: compute $X^2$ for 5,474 round vs 1,850 wrinkled under
3:1, its degrees of freedom and $p$-value. Then verify the two-category identity $X^2 = Z^2$ numerically for these counts.

---
## 3 · The χ² distribution

> 📍 **Where we are.** We used `stats.chi2` as a black box. Here is what is inside: a family of densities on the positive
> numbers, one for each number of degrees of freedom, built from squared standard normals.

If $X\sim\chi^2_p$ ("chi-squared with $p$ degrees of freedom"), its density is, for $x>0$,

$$p(X=x) \;=\; \frac{x^{\frac{p}{2}-1}}{\Gamma\!\left(\frac p2\right)2^{\frac p2}}\,e^{-\frac x2}, \qquad\text{and }0\text{ for }x\le0 .$$

*In words:* a power of $x$ times a decaying exponential — the same shape as the Gamma distribution (it is the
Gamma distribution with shape $p/2$ and rate $\tfrac12$); the constant with the Gamma function $\Gamma$ (see [Module 25 · 🚦 Hypothesis testing: errors, power, z- and t-tests](25_hypothesis_testing.ipynb)) makes
the total area 1. Here $p$ is the number of degrees of freedom — not a probability — and in all cases $x>0$, because the
statistic is a sum of squares.

**Where it comes from.** If $Z_1, Z_2,\dots,Z_p$ are independent standard normal random variables, then

$$\sum_{i=1}^p Z_i^2\ \sim\ \chi^2_p ,\qquad E\Big[\sum Z_i^2\Big] = p,\qquad \operatorname{Var}\Big[\sum Z_i^2\Big] = 2p .$$

*In words:* add up $p$ squared standard normals and you get a χ² variable with $p$ degrees of freedom; its mean is $p$
(each $Z_i^2$ has mean $E[Z^2] = \operatorname{Var}Z = 1$) and its variance is $2p$ (each $Z_i^2$ has variance
$E[Z^4]-1 = 3-1 = 2$, and variances of independent terms add — Module 21).

In [ ]:
# ── 3.1 The χ² density by hand vs SciPy, and the shapes for 1–5 degrees of freedom ──
def chi2_density(x, dof):
    """Density of the chi-squared distribution with `dof` degrees of freedom (formula above), for x > 0."""
    x = np.asarray(x, dtype=float)
    log_const = -(lgamma(dof / 2) + (dof / 2) * np.log(2))
    return np.exp(log_const + (dof / 2 - 1) * np.log(x) - x / 2)


xs = np.linspace(0.02, 10, 500)
for dof in [1, 2, 3, 5]:
    print(f"df = {dof}: density at x = 0.5, 1, 4 by hand {np.round(chi2_density([0.5, 1, 4], dof), 5)} "
          f"| scipy {np.round(stats.chi2.pdf([0.5, 1, 4], dof), 5)}")

fig, ax = plt.subplots(figsize=(6.4, 3.6))
colours = ["tab:blue", "tab:green", "tab:red", "tab:orange", "tab:purple"]
for dof, col in zip(range(1, 6), colours):
    ax.plot(xs, chi2_density(xs, dof), color=col, label=f"χ²$_{dof}$")
ax.set_ylim(0, 0.8); ax.set_xlim(0, 10)
ax.set_xlabel("x"); ax.set_ylabel("density p(x)"); ax.legend()
ax.set_title("χ² densities with 1–5 degrees of freedom")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Our formula agrees with `scipy.stats.chi2.pdf` at every point we printed. The shapes change a
lot with the degrees of freedom: with 1 df the density shoots up to infinity at $x\to0$ (a squared normal is usually
small); with 2 df it is the exponential $\tfrac12e^{-x/2}$ (density 0.5 at 0); from 3 df on it rises from 0 to a peak at
$x = p-2$ and then decays, with a long right tail. As $p$ grows, the peak moves right (the mean is $p$) and the curve
widens (the SD is $\sqrt{2p}$).

In [ ]:
# ── 3.2 A sum of p squared standard normals IS χ²_p (simulation) ──
rng_z = np.random.default_rng(SEED + 32)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, dof in zip(axes, [1, 3, 5]):
    sums = (rng_z.standard_normal((50_000, dof)) ** 2).sum(axis=1)
    ax.hist(sums, bins=80, range=(0, 20), density=True, color="0.75", label="Σ Z² (50,000 draws)")
    g = np.linspace(0.05, 20, 400)
    ax.plot(g, stats.chi2.pdf(g, dof), color="tab:red", label=f"χ²$_{dof}$ density")
    ax.set_ylim(0, 0.6 if dof > 1 else 1.2)
    ax.set_title(f"p = {dof}: mean {sums.mean():.2f} (theory {dof}), var {sums.var():.2f} (theory {2 * dof})", fontsize=9)
    ax.set_xlabel("sum of squared standard normals"); ax.legend(fontsize=7)
axes[0].set_ylabel("density")
plt.tight_layout(); plt.show()

🔍 **What just happened?** For $p$ = 1, 3 and 5, the histogram of $Z_1^2+\dots+Z_p^2$ lies on the $\chi^2_p$ density, and
the simulated means and variances (printed in the titles) are close to $p$ and $2p$. This is the bridge between the
goodness-of-fit statistic and the χ² table: $X^2$ is (approximately) a sum of $k-1$ squared standard normals.

### Reading a χ² table

As with the $t$ distribution, χ² tail probabilities used to be looked up in tables. We generate one: each entry is the
value $x^*$ with $P(\chi^2_\nu\ge x^*)$ = the column heading.

In [ ]:
# ── 3.3 Generate a χ² table: critical values x* with P(χ²_ν ≥ x*) = column heading ──
upper = [0.95, 0.9, 0.75, 0.5, 0.25, 0.1, 0.05, 0.025, 0.01, 0.005, 0.001]
dfs = list(range(1, 21)) + [25, 30, 40, 50, 60]
chi2_table = pd.DataFrame([[stats.chi2.isf(a, nu) for a in upper] for nu in dfs],
                          index=pd.Index(dfs, name="ν (df)"), columns=[str(a) for a in upper])
pd.set_option("display.width", 130)
print("upper-tail probabilities P(χ² ≥ x*) → critical values x*")
print(chi2_table.round(3).to_string())

🔍 **What just happened?** Row ν = 3 reads 0.352, 0.584, 1.213, 2.366, 4.108, 6.251, **7.815** (the 5 % cut-off), 9.348,
11.345, 12.838, 16.266. Note the columns on the left (0.95, 0.9, 0.75): χ² is not symmetric, so the lower tail needs its own
columns.

**Mendel with the table.** Our statistic is 0.47 with ν = 3. In row 3 it lies between 0.352 (column 0.95) and 0.584
(column 0.9), so $0.9 < p < 0.95$ — consistent with the exact $p = 0.925$. To *reject* at α = 0.05 we would need
$X^2\ge7.815$.

### "Too good to be true?" — the lower tail

A χ² statistic can also be suspiciously **small**: if data agree with a theory much more closely than random sampling
allows, one may wonder whether the data were selected or adjusted. For Mendel's dihybrid counts the lower-tail
probability is $P(\chi^2_3\le0.47) = 0.075$: a fit this good happens in about 7.5 % of honest experiments — unremarkable
on its own. In 1936 R. A. Fisher, one of the founders of modern statistics, pooled the χ² statistics of **all** of Mendel's
reported experiments and found that, taken together, they agreed with the expected ratios more closely than chance would
usually allow; he also argued that some of Mendel's expected ratios should have been different. Historians and
statisticians still debate the explanation — selective reporting of the clearest experiments, unconscious bias in
classifying borderline seeds, or an assistant who knew the expected answer — and many have defended Mendel's integrity;
nobody disputes that his laws are correct (Franklin *et al.* 2008). The lesson for us: the χ² distribution says how
much **disagreement** to expect, in both directions.

In [ ]:
# ── 3.4 Mendel's X² in both tails ──
print(f"upper tail  P(χ²_3 ≥ {x2:.3f}) = {stats.chi2.sf(x2, 3):.4f}   (evidence against the theory: none)")
print(f"lower tail  P(χ²_3 ≤ {x2:.3f}) = {stats.chi2.cdf(x2, 3):.4f}   (is the fit 'too good'? not on its own)")
print(f"fraction of the 20,000 simulated honest experiments that fit at least as well: {np.mean(sim_x2 <= x2):.4f}")

🔍 **What just happened?** The upper tail (0.925) says there is no evidence against 9:3:3:1; the lower tail (0.075),
confirmed by the simulated experiments, says a fit this close happens in about one honest experiment in thirteen. One
experiment cannot be "too good"; Fisher's argument rested on the combination of many.

### 📘 Lecture notes — the χ² distribution and χ² tables

- **Density** ($p$ = degrees of freedom, $x>0$): $p(X=x) = \dfrac{x^{p/2-1}}{\Gamma(p/2)\,2^{p/2}}e^{-x/2}$.
- **Construction:** if $Z_1,\dots,Z_p$ are independent standard normals, $\sum Z_i^2\sim\chi^2_p$; mean $p$, variance $2p$.
- **Shapes:** 1 df: infinite spike at 0; 2 df: exponential with density $\tfrac12$ at 0; ≥ 3 df: peak at $p-2$, long right
  tail; more df → shifted right and wider.
- **Table:** entries $x^*$ with $P(\chi^2_\nu\ge x^*)$ = column heading; 5 % cut-offs 3.841 (1 df), 5.991 (2), 7.815 (3),
  9.488 (4), 11.070 (5). Mendel: 0.47 lies between the 0.95 and 0.9 columns of row 3 → $p\approx0.925$.
- **Both tails:** large $X^2$ = poor fit; a very small $X^2$ (lower tail) = a fit closer than chance usually allows
  (Fisher's critique of Mendel's pooled data).

> 🎓 **Exam tip.** The χ² test of fit is one-sided in the statistic — reject for *large* $X^2$ — even though $H_1:\dot p\ne p$ is "two-sided" in the parameters. Use the upper-tail column of the table.

---
## 4 · Hardy–Weinberg equilibrium: a test with an estimated parameter

> 📍 **Where we are.** Mendel's theory fixed $\dot p$ completely. Often the theory has a free parameter that we must
> **estimate** from the same data — and each estimated parameter costs one degree of freedom.

**The biology.** At a SNP with alleles $A$ (frequency $1-p$) and $a$ (frequency $p$), random mating in a large population
combines two independently drawn alleles in each individual, so the genotypes $AA$, $Aa$, $aa$ (coded 0, 1, 2 copies of
$a$) have the **Hardy–Weinberg proportions**

$$P(0) = (1-p)^2,\qquad P(1) = 2p(1-p),\qquad P(2) = p^2 .$$

*In words:* both alleles must be $A$ for genotype 0; one of two orders ($Aa$ or $aA$) for genotype 1; both $a$ for
genotype 2 — the binomial distribution with 2 trials (Module 18). Departures from these proportions flag genotyping
errors (a standard quality-control step in every GWAS), inbreeding, population structure or selection.

**The test.** We do not know $p$, so we estimate it by maximum likelihood from the genotype counts $n_0, n_1, n_2$
($n = n_0+n_1+n_2$ individuals, $2n$ alleles) — the allele-counting estimate of [Module 23 · 🎯 Parameter estimation and maximum likelihood](23_maximum_likelihood.ipynb):

$$\hat p = \frac{n_1 + 2n_2}{2n} .$$

*In words:* count the $a$ alleles (one per heterozygote, two per $aa$ homozygote) and divide by the total number of
alleles. Then $E_0 = n(1-\hat p)^2$, $E_1 = 2n\hat p(1-\hat p)$, $E_2 = n\hat p^2$, and $X^2=\sum(O-E)^2/E$ with

$$\text{df} = (\text{categories} - 1) - (\text{parameters estimated}) = (3-1) - 1 = 1 .$$

*In words:* the estimate $\hat p$ was chosen to make the expected counts fit the observed ones as well as possible, which
removes one more free dimension from the deviations.

In [ ]:
# ── 4.1 Hardy–Weinberg tests: a random-mating sample, an inbred sample and the genotypes of Module 23 (SIMULATED) ──
def hwe_test(genotypes):
    """χ² test of Hardy–Weinberg proportions for 0/1/2 genotypes (1 df). Returns counts, p̂, expected, X², p."""
    counts = np.bincount(genotypes, minlength=3)
    n = counts.sum()
    p_hat = (counts[1] + 2 * counts[2]) / (2 * n)                      # MLE of the allele frequency
    exp_counts = n * np.array([(1 - p_hat) ** 2, 2 * p_hat * (1 - p_hat), p_hat ** 2])
    x2_hw = np.sum((counts - exp_counts) ** 2 / exp_counts)
    return counts, p_hat, exp_counts, x2_hw, stats.chi2.sf(x2_hw, df=1)


samples_hw = {"random mating (F = 0)": simulate_genotypes(400, 0.35),
              "inbred population (F = 0.15)": simulate_genotypes(400, 0.35, inbreeding=0.15),
              "the 500 genotypes of Module 23 (true p = 0.3)": simulate_genotypes(500, 0.3, seed=7)}
for label, g in samples_hw.items():
    counts, p_hat, e, x2_hw, p_hw = hwe_test(g)
    F_hat = 1 - counts[1] / e[1]                                       # observed vs expected heterozygosity
    print(f"{label}: counts (0, 1, 2) = {counts}, p̂ = {p_hat:.4f}")
    print(f"   expected = {np.round(e, 2)}, X² = {x2_hw:.3f}, df = 1, p = {p_hw:.4f}, "
          f"heterozygote deficit F̂ = 1 − O₁/E₁ = {F_hat:.3f}")
    print(f"   scipy.stats.chisquare(ddof=1): p = {stats.chisquare(counts, e, ddof=1).pvalue:.4f}")

🔍 **What just happened?** In the random-mating sample (400 individuals, $\hat p = 0.344$) the counts 168, 189, 43 are
close to the expected 172.3, 180.5, 47.3 (even a slight excess of heterozygotes, $\hat F = -0.047$): $X^2 = 0.894$,
$p = 0.344$ — consistent with Hardy–Weinberg. In the inbred
sample there are too few heterozygotes: the heterozygote deficit $\hat F\approx0.16$ (the simulation used $F=0.15$),
$X^2 = 10.1$ and $p = 0.0015$ — strong evidence against Hardy–Weinberg proportions. Inbreeding makes the two alleles of an
individual more likely to be identical copies of the same ancestral allele, so it moves individuals from $Aa$ into $AA$ and
$aa$. `scipy.stats.chisquare(..., ddof=1)` (one estimated parameter) gives the same $p$-values.

The third sample is the one of [Module 23 · 🎯 Parameter estimation and maximum likelihood](23_maximum_likelihood.ipynb) (500 people at one SNP, counts 231, 226, 43), where we noticed a few more
heterozygotes than Hardy–Weinberg predicts (226 observed, 214.7 expected). Now we can say how surprising that is:
$\hat p = 0.312$, $X^2 = 1.40$ and $p = 0.237$ — an excess this small is ordinary sampling noise, so those genotypes are
consistent with Hardy–Weinberg proportions.

Why exactly 1 degree of freedom and not 2? Let us check by simulation: many samples from a population that *is* in
Hardy–Weinberg equilibrium, each tested with its own estimated $\hat p$.

In [ ]:
# ── 4.2 Estimating p costs one degree of freedom (simulation under H0) ──
rng_hw = np.random.default_rng(SEED + 42)
p_true, n_ind = 0.35, 400
probs_hw = np.array([(1 - p_true) ** 2, 2 * p_true * (1 - p_true), p_true ** 2])
cnt = rng_hw.multinomial(n_ind, probs_hw, size=20_000)
p_hats = (cnt[:, 1] + 2 * cnt[:, 2]) / (2 * n_ind)
E_hat = n_ind * np.column_stack([(1 - p_hats) ** 2, 2 * p_hats * (1 - p_hats), p_hats ** 2])
x2_est = ((cnt - E_hat) ** 2 / E_hat).sum(axis=1)                      # p estimated from each sample
x2_known = ((cnt - n_ind * probs_hw) ** 2 / (n_ind * probs_hw)).sum(axis=1)   # true p plugged in

g = np.linspace(0.02, 12, 300)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.hist(x2_est, bins=60, range=(0, 12), density=True, alpha=0.55, color="tab:blue", label="p̂ estimated from the sample")
ax.hist(x2_known, bins=60, range=(0, 12), density=True, alpha=0.45, color="tab:orange", label="true p known")
ax.plot(g, stats.chi2.pdf(g, 1), color="tab:blue", lw=2, label="χ²₁")
ax.plot(g, stats.chi2.pdf(g, 2), color="tab:orange", lw=2, ls="--", label="χ²₂")
ax.set_ylim(0, 1.0); ax.set_xlabel("X²"); ax.set_ylabel("density"); ax.legend()
ax.set_title("Hardy–Weinberg X² under H0: 20,000 simulated samples")
plt.tight_layout(); plt.show()
print(f"mean X² with p̂: {x2_est.mean():.3f} (χ²₁ mean 1);  with the true p: {x2_known.mean():.3f} (χ²₂ mean 2)")
print(f"rejection rate at the χ²₁ 5% cut-off 3.841: {np.mean(x2_est >= 3.841):.4f}")

🔍 **What just happened?** When the true $p$ is plugged in, $X^2$ follows $\chi^2_2$ (mean ≈ 2): three categories minus
one constraint. When $p$ is estimated from each sample, $X^2$ follows $\chi^2_1$ (mean ≈ 1), and rejecting at the $\chi^2_1$
cut-off 3.841 gives a Type I error rate of about 5 %. Fitting a parameter pulls the expected counts towards the observed
ones and removes one degree of freedom; using 2 df here would make the test too conservative (it would rarely reject).

✏️ **Your turn.** Simulate `simulate_genotypes(400, 0.35, inbreeding=F)` for F = 0.02, 0.05, 0.10 and 0.20 with several
seeds. How large must the inbreeding coefficient be before the test detects it reliably with 400 people? With 4,000?

---
## 5 · Contingency tables: is a genotype associated with a disease?

> 📍 **Where we are.** So far one categorical variable was compared with a theory. Now we cross **two** categorical
> variables — disease status and genotype — and ask whether they are **independent** (Modules 17 and 19). This is the
> logic of a case–control genetic association study.

A **contingency table** counts individuals by two categorical variables. Rows: $r$ disease groups; columns: $c$ genotypes.
Write $O_{ij}$ for the count in row $i$, column $j$, $R_i$ for the row totals, $C_j$ for the column totals and $n$ for the
grand total. We test

$$H_0:\ \text{disease status and genotype are independent},\qquad\text{i.e.}\qquad P(\text{row } i, \text{column } j) = P(\text{row } i)\,P(\text{column } j)\ \text{ for all } i, j .$$

**Expected counts under independence.** Estimate the marginal probabilities by the observed fractions,
$\hat P(\text{row } i) = R_i/n$ and $\hat P(\text{column } j) = C_j/n$. Under $H_0$ the cell probability is their product,
so the expected count is

$$E_{ij} = n\cdot\frac{R_i}{n}\cdot\frac{C_j}{n} = \frac{R_i\,C_j}{n} \qquad\Big(\text{row total} \times\text{column total} \,/\, n\Big).$$

*In words:* if genotype has nothing to do with disease, each genotype column should be split between cases and controls in
the same proportions as the whole sample. The statistic is again $X^2 = \sum_{i,j}(O_{ij}-E_{ij})^2/E_{ij}$, with

$$\text{df} = (rc - 1) - \big[(r-1)+(c-1)\big] = (r-1)(c-1) .$$

*In words:* $rc$ cells, minus 1 for the fixed total, minus the $(r-1)+(c-1)$ marginal probabilities we estimated. For a
$2\times3$ table: $(2-1)(3-1) = 2$ degrees of freedom.

In [ ]:
# ── 5.1 A case–control study at one SNP: 500 cases, 500 controls (SIMULATED) ──
geno_cases = simulate_genotypes(500, 0.38, seed=2)        # risk-allele frequency 0.38 in cases …
geno_controls = simulate_genotypes(500, 0.30, seed=1)     # … and 0.30 in controls (the hidden truth)
table = np.array([np.bincount(geno_cases, minlength=3), np.bincount(geno_controls, minlength=3)])
row_tot, col_tot, n_tot = table.sum(axis=1), table.sum(axis=0), table.sum()
E_tab = np.outer(row_tot, col_tot) / n_tot                # E_ij = R_i C_j / n
x2_tab = ((table - E_tab) ** 2 / E_tab).sum()
df_tab = (table.shape[0] - 1) * (table.shape[1] - 1)
print(pd.DataFrame(table, index=["cases", "controls"], columns=["0 copies", "1 copy", "2 copies"]).assign(total=row_tot))
print("column totals:", col_tot, " n =", n_tot)
print("expected counts under independence:\n", pd.DataFrame(E_tab, index=["cases", "controls"],
                                                           columns=["0 copies", "1 copy", "2 copies"]).round(2))
print(f"X² = {x2_tab:.3f}, df = (2−1)(3−1) = {df_tab}, p = {stats.chi2.sf(x2_tab, df_tab):.5f}")
res = stats.chi2_contingency(table)
print(f"scipy.stats.chi2_contingency: X² = {res.statistic:.3f}, df = {res.dof}, p = {res.pvalue:.5f}")

🔍 **What just happened?** Cases carry the risk allele more often: 70 cases vs 46 controls have two copies, 198 cases vs
255 controls have none. Under independence each column would be split 50 : 50 (there are 500 cases and 500 controls),
giving expected counts 226.5, 215.5 and 58 in both rows. The deviations add up to **X² = 14.66** on 2 df, $p = 0.00065$:
strong evidence that genotype and disease status are associated. `chi2_contingency` reproduces every number (for tables
larger than $2\times2$ it applies no continuity correction).

**The allelic test.** Genetic association studies more often count **alleles** instead of people: each person contributes
two alleles, giving a $2\times2$ table (cases/controls × risk/other allele) with 1 degree of freedom. Its effect size is
the **odds ratio**: the odds of carrying the risk allele in cases divided by the odds in controls.

In [ ]:
# ── 5.2 The allelic 2 × 2 test and the odds ratio ──
def allele_counts(row):
    """(risk alleles, other alleles) from genotype counts (n0, n1, n2)."""
    return np.array([row[1] + 2 * row[2], 2 * row[0] + row[1]])


allelic = np.array([allele_counts(table[0]), allele_counts(table[1])])
print(pd.DataFrame(allelic, index=["cases", "controls"], columns=["risk allele", "other allele"]))
E_al = np.outer(allelic.sum(axis=1), allelic.sum(axis=0)) / allelic.sum()
x2_al = ((allelic - E_al) ** 2 / E_al).sum()
odds_ratio = (allelic[0, 0] / allelic[0, 1]) / (allelic[1, 0] / allelic[1, 1])
print(f"allelic X² = {x2_al:.3f}, df = 1, p = {stats.chi2.sf(x2_al, 1):.2e}; odds ratio = {odds_ratio:.3f}")
print("scipy (no continuity correction):", stats.chi2_contingency(allelic, correction=False)[:2])

🔍 **What just happened?** Counting alleles, cases carry 372 risk alleles out of 1,000 and controls 291 out of 1,000. The
$2\times2$ test gives **X² = 14.80** on 1 df ($p = 1.2\times10^{-4}$) and an odds ratio of **1.44**: the odds of a risk
allele are 44 % higher in cases. This single-SNP test, repeated at a million SNPs, is the basic engine of a case–control
genome-wide association study (GWAS).

### Small counts: Fisher's exact test

The χ² approximation needs large expected counts. With small samples — a rare variant, a small clinical cohort — use
**Fisher's exact test**. For a $2\times2$ table with fixed margins, under independence the count in the top-left cell
follows a **hypergeometric distribution** (Section 9 derives it by counting), and the exact $p$-value adds up the
probabilities of all tables at least as extreme as the observed one. *Illustrative example:* a rare variant is carried by
7 of 10 patients with a rare disease and 1 of 12 healthy relatives.

In [ ]:
# ── 5.3 Fisher's exact test vs the χ² approximation on a small table (illustrative counts) ──
small = np.array([[7, 3],     # patients: carriers, non-carriers
                  [1, 11]])   # relatives: carriers, non-carriers
E_small = np.outer(small.sum(axis=1), small.sum(axis=0)) / small.sum()
print("expected counts under independence:\n", E_small.round(2))
chi_small = stats.chi2_contingency(small, correction=False)
fisher = stats.fisher_exact(small)
print(f"χ² approximation: X² = {chi_small[0]:.3f}, p = {chi_small[1]:.4f}  (expected counts below 5 → unreliable)")
print(f"Fisher's exact test: odds ratio = {fisher.statistic:.2f}, exact two-sided p = {fisher.pvalue:.4f}")

🔍 **What just happened?** Two of the four expected counts (3.64 and 4.36) are below 5, so the χ² approximation is
doubtful; it gives $p = 0.0028$. Fisher's exact test, which needs no approximation, gives $p = 0.0062$ — still strong
evidence of association, but more than twice as large: here the approximation overstates the evidence. For small tables,
use the exact test.

**What "exact" means.** Fix the margins: 10 patients and 12 relatives, 8 carriers and 14 non-carriers, 22 people. Under
independence, which 8 of the 22 people are the carriers is a random choice, so the number of carriers among the 10
patients, $X$, is hypergeometric (Section 9 derives the formula by counting):
$P(X=k) = \binom{8}{k}\binom{14}{10-k}\big/\binom{22}{10}$. The two-sided $p$-value adds the probabilities of all tables
that are **at most as probable** as the observed one ($k = 7$).

In [ ]:
# ── 5.4 Fisher's exact test by hand: enumerate every table with the same margins ──
n_pat, n_carriers, n_all = 10, 8, 22
probs_k = {k: comb(n_carriers, k) * comb(n_all - n_carriers, n_pat - k) / comb(n_all, n_pat)
           for k in range(max(0, n_pat + n_carriers - n_all), min(n_pat, n_carriers) + 1)}
p_obs_table = probs_k[7]
for k, pk in probs_k.items():
    flag = "  ← observed" if k == 7 else ("  ← counted (at most as probable)" if pk <= p_obs_table * (1 + 1e-9) else "")
    print(f"carrier patients k = {k}: P(X = k) = {pk:.5f}{flag}")
p_exact = sum(pk for pk in probs_k.values() if pk <= p_obs_table * (1 + 1e-9))
print(f"the probabilities sum to {sum(probs_k.values()):.6f};  exact two-sided p = {p_exact:.4f} "
      f"(scipy.stats.fisher_exact: {stats.fisher_exact(small).pvalue:.4f})")

🔍 **What just happened?** With the margins fixed, the number of carrier patients can only be 0, 1, …, 8, and the nine
probabilities sum to 1. The observed table ($k=7$) has probability 0.00450. Two other tables are at most as probable: the
even more extreme $k = 8$ (0.00014) and, in the opposite direction, $k = 0$ (0.00155 — no patient carries the variant).
Their sum is the exact two-sided $p = 0.0062$, exactly SciPy's value. No normal or χ² approximation was used — only
counting.

### 📘 Lecture notes — Hardy–Weinberg and contingency tables

- **Estimated parameters cost degrees of freedom:** df = (categories − 1) − (number of parameters estimated from the data).
- **Hardy–Weinberg test:** $\hat p = (n_1+2n_2)/(2n)$; expected $n(1-\hat p)^2,\ 2n\hat p(1-\hat p),\ n\hat p^2$;
  $X^2$ on **1 df**. Heterozygote deficit (inbreeding, population structure, genotyping error) → large $X^2$.
- **Independence in an $r\times c$ table:** $E_{ij} = R_iC_j/n$ (row total × column total / $n$), $X^2 = \sum(O-E)^2/E$,
  df = $(r-1)(c-1)$. Genotype × disease ($2\times3$): 2 df; allelic $2\times2$: 1 df, effect size = odds ratio.
- **Fisher's exact test** (hypergeometric) when expected counts are small (rule of thumb: any $E_{ij}<5$).

> 🎓 **Exam tip.** Show the expected-count formula $R_iC_j/n$ as the product of two estimated marginal probabilities times $n$ — that is where the independence hypothesis enters.

### 🌉 Bridge — Genome-wide association studies

A GWAS repeats the test of Section 5.2 (or a regression version of it, [Module 27 · 📉 Linear regression: the whole course in one model](27_linear_regression_review.ipynb)) at roughly a million SNPs. Because so many
tests are run, the conventional **genome-wide significance threshold** is $p < 5\times10^{-8}$ — a Bonferroni correction
(Section 6) of α = 0.05 for about one million effectively independent common variants in the human genome. The
Hardy–Weinberg test of Section 4 is used for quality control: SNPs far out of equilibrium in the controls are usually
genotyping artefacts and are removed before the association scan. Regression-based association tests, Manhattan plots and
polygenic scores are developed in [BiologyToML Module 06 · Linear regression and GWAS](../../../../ml/tutorial/BiologyToML/notebooks/06_linear_regression_gwas.ipynb)
and [Module 35 · Statistical genetics](../../../../ml/tutorial/BiologyToML/notebooks/35_statistical_genetics.ipynb).

✏️ **Your turn.** Halve the study (250 cases and 250 controls: `simulate_genotypes(250, …)`) and rerun 5.1–5.2. Would this
SNP pass the genome-wide threshold $5\times10^{-8}$ in either study? What does that tell you about the sample sizes of
modern GWAS?

---
## 6 · The multiple-testing problem

> 📍 **Where we are.** One test at α = 0.05 is wrong 5 % of the time when $H_0$ is true. A genomics experiment runs one test
> per gene — thousands at once. What happens to the error rates?

**Expected number of false positives.** Suppose we run $m$ tests and all $m$ null hypotheses are true. Let $V$ be the number
of (false) rejections. Each test rejects with probability α, so by linearity of expectation (Module 21 — no independence
needed)

$$E[V] = \sum_{i=1}^m P(\text{test } i \text{ rejects}) = m\,\alpha .$$

*In words:* 20,000 genes tested at α = 0.05 give about 1,000 "significant" genes even if nothing changed.

**Family-wise error rate (FWER)** = the probability of **at least one** false positive. If the $m$ tests are independent,
by the complement rule (Module 16):

$$\text{FWER} = P(V\ge1) = 1-P(V=0) = 1-\prod_{i=1}^m P(\text{test } i \text{ does not reject}) = 1-(1-\alpha)^m .$$

*In words:* the chance that *every* test avoids a false positive is $(1-\alpha)^m$, which shrinks fast as $m$ grows.

In [ ]:
# ── 6.1 Expected false positives and the FWER as the number of tests grows ──
alpha = 0.05
print(f"{'m tests':>8s} {'E[V] = mα':>10s} {'FWER = 1−(1−α)^m':>18s}")
for m in [1, 5, 10, 20, 100, 400, 20_000]:
    print(f"{m:8d} {m * alpha:10.2f} {1 - (1 - alpha) ** m:18.6f}")

ms = np.arange(1, 201)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(ms, 1 - (1 - alpha) ** ms, label="no correction: 1 − (1 − α)^m")
ax.plot(ms, 1 - (1 - alpha / ms) ** ms, label="Bonferroni (each test at α/m)")
ax.axhline(alpha, color="0.5", ls=":", lw=1)
ax.set_xlabel("number of tests m (all nulls true)"); ax.set_ylabel("P(at least one false positive)")
ax.set_title("family-wise error rate, α = 0.05"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** With 10 tests the chance of at least one false positive is already **40 %**, with 100 tests
**99.4 %**, and with 400 tests (our infection experiment) it is essentially certain — while about $400\times0.05 = 20$
false positives are expected. The orange curve shows the fix below: testing each hypothesis at level α/m keeps the FWER at
or below 0.05 (it approaches $1-e^{-0.05}\approx0.049$) however many tests are run.

### Bonferroni correction

**Rule:** reject $H_{0,i}$ only if $p_i\le\alpha/m$. **Claim:** $\text{FWER}\le\alpha$, whatever the dependence between the
tests. **Proof** with the union bound (Boole's inequality, a consequence of inclusion–exclusion, Module 16:
$P(A\cup B)\le P(A)+P(B)$). Let $A_i$ be the event "test $i$ rejects although $H_{0,i}$ is true" and $m_0\le m$ the number of
true nulls:

$$\text{FWER} = P\Big(\bigcup_{i\,:\,H_{0,i}\text{ true}} A_i\Big) \;\le\; \sum_{i\,:\,H_{0,i}\text{ true}} P(A_i) \;=\; m_0\,\frac{\alpha}{m} \;\le\; \alpha .$$

*In words:* the probability that at least one of the true nulls is falsely rejected is at most the sum of their individual
probabilities, each of which is α/m. Bonferroni is simple and always valid — but very strict: with 20,000 genes each gene
needs $p\le2.5\times10^{-6}$, and many real effects are missed.

In [ ]:
# ── 6.2 A simulated screen: 10,000 tests, 1,000 real effects (SIMULATED z-statistics) ──
rng_scr = np.random.default_rng(SEED + 62)
m, m1 = 10_000, 1_000
is_real = np.zeros(m, dtype=bool); is_real[:m1] = True
z_scr = rng_scr.normal(0, 1, m) + np.where(is_real, 3.0, 0.0)     # real effects shift z by 3 standard errors
p_scr = 2 * stats.norm.sf(np.abs(z_scr))


def summarise(reject, truth):
    """Discoveries R, false discoveries V, true discoveries, false discovery proportion V/R and power."""
    R, V = reject.sum(), (reject & ~truth).sum()
    return {"discoveries R": R, "false V": V, "true": R - V, "V/R": V / max(R, 1), "power": (reject & truth).sum() / truth.sum()}


rows = {"no correction (p ≤ 0.05)": summarise(p_scr <= 0.05, is_real),
        "Bonferroni (p ≤ 0.05/m)": summarise(p_scr <= 0.05 / m, is_real)}
print(pd.DataFrame(rows).T.round(3).to_string())

🔍 **What just happened?** Without correction the screen "discovers" 1,287 hits, of which 428 — a third — are false
(the expected number is $9{,}000\times0.05 = 450$). Bonferroni removes all false positives but also discards most real
effects: it finds only 56 of the 1,000, a power of 5.6 %. Neither extreme is what a biologist wants from a screen. The
next section introduces the compromise used throughout genomics.

**What if the tests are dependent?** Genes in a pathway, or neighbouring SNPs, give correlated test statistics. The
formula $1-(1-\alpha)^m$ assumed independence; the Bonferroni bound did not. Let us simulate 100 null tests whose
$z$-statistics share a common component, $Z_i = \sqrt\rho\,W + \sqrt{1-\rho}\,\varepsilon_i$ (each $Z_i$ is still
$\mathcal N(0,1)$, and any two have correlation $\rho$).

In [ ]:
# ── 6.3 FWER with correlated tests: 100 null tests, 20,000 simulated experiments ──
rng_dep = np.random.default_rng(SEED + 63)
m_dep, n_exp = 100, 20_000
print(f"{'correlation ρ':>14s} {'FWER, no correction':>20s} {'FWER, Bonferroni':>17s}")
for rho in [0.0, 0.5, 0.9]:
    shared = rng_dep.standard_normal((n_exp, 1))
    z_dep = np.sqrt(rho) * shared + np.sqrt(1 - rho) * rng_dep.standard_normal((n_exp, m_dep))
    p_dep = 2 * stats.norm.sf(np.abs(z_dep))
    fwer_raw = np.mean((p_dep <= 0.05).any(axis=1))
    fwer_bonf = np.mean((p_dep <= 0.05 / m_dep).any(axis=1))
    print(f"{rho:14.1f} {fwer_raw:20.4f} {fwer_bonf:17.4f}")
print(f"independence formula 1 − 0.95^100 = {1 - 0.95 ** 100:.4f}")

🔍 **What just happened?** With independent tests ($\rho=0$) the simulated FWER without correction matches the formula
(0.9944 vs 0.9941) and Bonferroni holds it just below 0.05 (0.045). As the correlation grows, the tests increasingly
reject *together* or not at all, so the uncorrected FWER falls (0.78 at $\rho = 0.5$, 0.22 at $\rho=0.9$), and Bonferroni
becomes **conservative**: its FWER drops to 0.028 and then 0.004 —
it is still valid (the union bound never fails), but it pays for 100 tests when the data contain fewer "effective"
independent tests. This is why GWAS use a threshold based on the *effective* number of independent SNPs (≈ one million)
rather than the number genotyped.

### 📘 Lecture notes — multiple testing and the family-wise error rate

- $m$ tests, all nulls true: expected false positives $E[V] = m\alpha$ (linearity of expectation).
- **FWER** $=P(V\ge1)$; for independent tests $1-(1-\alpha)^m$ (0.40 for $m=10$, 0.994 for $m=100$).
- **Bonferroni:** reject if $p_i\le\alpha/m$. Then FWER $\le m_0\alpha/m\le\alpha$ by the union bound, for **any**
  dependence between tests. Very conservative for large $m$ (low power).
- Genome-wide significance $5\times10^{-8}$ ≈ 0.05 / 1,000,000 (Bonferroni for about a million independent common SNPs).

> 🎓 **Exam tip.** Be able to derive $1-(1-\alpha)^m$ (complement rule + independence) and the Bonferroni bound (union bound, no independence needed) — and to say which assumption each one uses.

---
## 7 · The false discovery rate and the Benjamini–Hochberg procedure

> 📍 **Where we are.** Bonferroni protects against *any* false positive. In an exploratory screen we can tolerate some
> false positives, as long as they are a small **fraction** of what we report. That fraction is the false discovery rate.

For a procedure that rejects $R$ of the $m$ hypotheses, $V$ of them true nulls, the **false discovery proportion** is
$V/R$ (defined as 0 when $R=0$), and the **false discovery rate** is its expectation:

$$\text{FDR} = E\left[\frac{V}{\max(R,1)}\right].$$

*In words:* on average, what fraction of the genes on my list are false? Controlling the FDR at $q = 0.05$ means that, over
many experiments, about 5 % of reported discoveries are false — a guarantee about the list, not about each gene.

**The Benjamini–Hochberg (BH) procedure** at level $q$:

1. sort the $p$-values: $p_{(1)}\le p_{(2)}\le\dots\le p_{(m)}$;
2. find the **largest** $k$ such that $p_{(k)}\le\dfrac{k}{m}\,q$;
3. reject the hypotheses with the $k$ smallest $p$-values (none if no such $k$ exists).

*In words:* compare the $k$-th smallest $p$-value with a threshold that grows linearly with its rank, from $q/m$ (the
Bonferroni threshold) for the smallest to $q$ for the largest; then cut the list at the last place where a $p$-value is
below its line. Benjamini and Hochberg (1995) proved that for independent tests this controls
$\text{FDR}\le\frac{m_0}{m}q\le q$.

⚠️ **Step-up, not step-down.** Step 2 looks for the *largest* passing rank. A $p$-value can be rejected even though it lies
above its own threshold, if a larger $p$-value further down the list passes.

**Worked example** ($m = 10$, $q = 0.05$, thresholds $k\cdot0.005$): sorted $p$ = 0.0008, 0.009, 0.018, 0.019, 0.020,
0.13, 0.27, 0.41, 0.66, 0.92.

In [ ]:
# ── 7.1 Benjamini–Hochberg by hand on the worked example ──
p_ex = np.array([0.0008, 0.009, 0.018, 0.019, 0.020, 0.13, 0.27, 0.41, 0.66, 0.92])
m_ex, q = p_ex.size, 0.05
thresholds = np.arange(1, m_ex + 1) / m_ex * q
passes = p_ex <= thresholds
k_max = np.max(np.nonzero(passes)[0]) + 1 if passes.any() else 0
print(pd.DataFrame({"rank k": np.arange(1, m_ex + 1), "p_(k)": p_ex, "threshold k·q/m": thresholds,
                    "p ≤ threshold?": passes, "BH rejects": np.arange(1, m_ex + 1) <= k_max}).to_string(index=False))
print(f"largest passing rank k = {k_max} → BH rejects {k_max} hypotheses;  "
      f"Bonferroni (p ≤ {q / m_ex}) rejects {np.sum(p_ex <= q / m_ex)}")

🔍 **What just happened?** Ranks 1 and 2 pass (0.0008 ≤ 0.005, 0.009 ≤ 0.010); rank 3 fails (0.018 > 0.015) — but ranks 4
and 5 pass again (0.019 ≤ 0.020, 0.020 ≤ 0.025), and nothing passes after rank 5. The largest passing rank is 5, so BH
rejects the **five** smallest $p$-values, including the third one that failed its own threshold (step-up). Bonferroni
(threshold 0.005) would reject only one.

**Adjusted $p$-values.** Software usually reports BH-adjusted $p$-values (often called $q$-values):
$\tilde p_{(k)} = \min_{j\ge k}\big(\frac{m}{j}p_{(j)}\big)$, capped at 1; rejecting all genes with $\tilde p\le q$ is the
same as running BH at level $q$. We implement both forms and check that they agree.

In [ ]:
# ── 7.2 A reusable BH implementation, two ways ──
def benjamini_hochberg(pvals, q=0.05):
    """Boolean array: which hypotheses the Benjamini–Hochberg step-up procedure rejects at FDR level q."""
    pvals = np.asarray(pvals, dtype=float)
    m = pvals.size
    order = np.argsort(pvals)
    below = pvals[order] <= np.arange(1, m + 1) / m * q
    reject = np.zeros(m, dtype=bool)
    if below.any():
        k = np.max(np.nonzero(below)[0])                  # largest passing rank (0-based)
        reject[order[:k + 1]] = True
    return reject


def bh_adjusted(pvals):
    """BH-adjusted p-values: p̃_(k) = min over j ≥ k of (m/j)·p_(j), capped at 1, in the original order."""
    pvals = np.asarray(pvals, dtype=float)
    m = pvals.size
    order = np.argsort(pvals)
    scaled = pvals[order] * m / np.arange(1, m + 1)
    adj_sorted = np.minimum(np.minimum.accumulate(scaled[::-1])[::-1], 1.0)
    adj = np.empty(m)
    adj[order] = adj_sorted
    return adj


print("adjusted p-values of the worked example:", np.round(bh_adjusted(p_ex), 4))
print("same decisions both ways:", np.array_equal(benjamini_hochberg(p_ex), bh_adjusted(p_ex) <= 0.05))
print("same decisions on the 10,000-test screen:",
      np.array_equal(benjamini_hochberg(p_scr), bh_adjusted(p_scr) <= 0.05))
if hasattr(stats, "false_discovery_control"):                # SciPy ≥ 1.11 has BH built in
    print("max |ours − scipy.stats.false_discovery_control|:",
          np.abs(bh_adjusted(p_scr) - stats.false_discovery_control(p_scr)).max())

rows["Benjamini–Hochberg (q = 0.05)"] = summarise(benjamini_hochberg(p_scr), is_real)
print(pd.DataFrame(rows).T.round(3).to_string())

🔍 **What just happened?** The adjusted $p$-values of the worked example are 0.008 for the first and **0.04** for ranks
2–5 (the running minimum from the bottom carries 0.04 upwards), then 0.2167 and larger: exactly five are ≤ 0.05, as by
hand. The step-up search and the adjusted $p$-values give identical decisions on the 10,000-test screen, and agree with
SciPy's built-in `false_discovery_control` up to floating-point round-off. On the screen BH reports 523 discoveries, of
which 17 are false (a false discovery proportion of 3.3 %) and 506 true (power 0.51) — nine times Bonferroni's power, and
25 times fewer false hits than no correction.

Is the "≈ 5 %" luck? The FDR is an *average* over experiments, so let us repeat the whole screen 200 times.

In [ ]:
# ── 7.3 Does BH control the FDR? 200 repeated screens (SIMULATED) ──
rng_rep = np.random.default_rng(SEED + 73)
fdp = {"none": [], "Bonferroni": [], "BH": []}
power = {"none": [], "Bonferroni": [], "BH": []}
for _ in range(200):
    z_r = rng_rep.normal(0, 1, m) + np.where(is_real, 3.0, 0.0)
    p_r = 2 * stats.norm.sf(np.abs(z_r))
    for name, rej in [("none", p_r <= 0.05), ("Bonferroni", p_r <= 0.05 / m), ("BH", benjamini_hochberg(p_r))]:
        fdp[name].append((rej & ~is_real).sum() / max(rej.sum(), 1))
        power[name].append((rej & is_real).sum() / m1)
print(f"{'procedure':12s} {'FDR (mean V/R)':>15s} {'mean power':>11s}")
for name in fdp:
    print(f"{name:12s} {np.mean(fdp[name]):15.4f} {np.mean(power[name]):11.4f}")
print(f"BH guarantee: FDR ≤ (m0/m)·q = {(m - m1) / m * 0.05:.4f}")

fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.hist(fdp["BH"], bins=25, color="tab:green", edgecolor="white")
ax.axvline(np.mean(fdp["BH"]), color="k", ls="--", label=f"mean = {np.mean(fdp['BH']):.3f}")
ax.axvline(0.05, color="tab:red", ls=":", label="q = 0.05")
ax.set_xlabel("false discovery proportion V/R of one screen"); ax.set_ylabel("number of screens")
ax.set_title("BH at q = 0.05: 200 simulated screens"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** Averaged over 200 screens, the false discovery proportion of BH is **0.0451** — right at the
theoretical value $\frac{m_0}{m}q = 0.9\times0.05 = 0.045$ — while its power is about **50 %**, more than eight times
Bonferroni's 5.9 %. The histogram shows that the FDP of a *single* screen varies (from about 0.02 to 0.08): FDR control is
a statement about the average, not a promise for one list. Without correction, about a third of all discoveries are false.

### 📘 Lecture notes — the false discovery rate and Benjamini–Hochberg

- **FDP** $= V/R$ (0 if $R=0$); **FDR** $= E[V/\max(R,1)]$ — the expected fraction of false discoveries among the reported
  ones. FWER control ⇒ FDR control, not conversely; FDR is the right target for exploratory screens.
- **BH at level $q$:** sort $p_{(1)}\le\dots\le p_{(m)}$; find the largest $k$ with $p_{(k)}\le kq/m$; reject the $k$
  smallest. **Step-up**: earlier ranks are rejected even if they miss their own threshold.
- **Adjusted $p$-values:** $\tilde p_{(k)} = \min_{j\ge k}\frac{m}{j}p_{(j)}$ (capped at 1); reject if $\tilde p\le q$.
- **Guarantee** (independent tests): $\text{FDR}\le\frac{m_0}{m}q$. In our simulation: FDR ≈ 0.045, power ≈ 0.50 vs
  Bonferroni ≈ 0.06.
- **Worked example** ($m=10$, $q=0.05$): $p$ = 0.0008, 0.009, 0.018, 0.019, 0.020, … → $k=5$ rejections (Bonferroni: 1).

> 🎓 **Exam tip.** Run BH by hand on a short list: write the thresholds $kq/m$ next to the sorted $p$-values and look for the **last** pass, not the first failure.

✏️ **Your turn.** Make the real effects weaker (shift 2 instead of 3) and stronger (shift 4) in 7.3. How do the FDR and
the power of BH change? Which of the two does BH keep fixed?

---
## 8 · A genome-wide screen: which genes respond to infection?

> 📍 **Where we are.** We have every ingredient: Welch's $t$-test for one gene ([Module 25 · 🚦 Hypothesis testing: errors, power, z- and t-tests](25_hypothesis_testing.ipynb)), and BH for many. Now the full
> pipeline on the course's infection experiment — 400 genes in spleen, kidney and liver, 4 control vs 4 infected samples
> each — where the simulator tells us which genes truly changed.

In [ ]:
# ── 8.1 Welch's t-test for all 400 genes in each tissue, vectorised (SIMULATED expression) ──
expr = simulate_tissue_expression()
X, genes, samples = expr["X"], np.array(expr["genes"]), expr["samples"]
tissues = ["spleen", "kidney", "liver"]


def welch_all_genes(A, B):
    """Welch t-tests of every column (gene): A, B are (samples × genes). Returns (difference of means, t, p)."""
    va, vb = A.var(axis=0, ddof=1) / A.shape[0], B.var(axis=0, ddof=1) / B.shape[0]
    t = (A.mean(axis=0) - B.mean(axis=0)) / np.sqrt(va + vb)
    df = (va + vb) ** 2 / (va ** 2 / (A.shape[0] - 1) + vb ** 2 / (B.shape[0] - 1))
    return A.mean(axis=0) - B.mean(axis=0), t, 2 * stats.t.sf(np.abs(t), df)


de = {}
for tissue in tissues:
    inf = X[((samples["tissue"] == tissue) & (samples["condition"] == "infected")).to_numpy()]
    con = X[((samples["tissue"] == tissue) & (samples["condition"] == "control")).to_numpy()]
    lfc, t_g, p_g = welch_all_genes(inf, con)
    truth = np.isin(genes, list(expr["true_de"][tissue]))
    de[tissue] = pd.DataFrame({"gene": genes, "log2FC": lfc, "t": t_g, "p": p_g, "p_BH": bh_adjusted(p_g),
                               "truly_changed": truth})
    check = stats.ttest_ind(inf, con, equal_var=False, axis=0).pvalue
    print(f"{tissue:6s}: {truth.sum():2d} truly changed genes of 400;  max |our p − scipy p| = {np.abs(p_g - check).max():.1e}")
print(de["spleen"].sort_values("p").head(8).round(5).to_string(index=False))

🔍 **What just happened?** One vectorised function ran 1,200 Welch tests (400 genes × 3 tissues) and matches
`scipy.stats.ttest_ind` to floating-point precision. The simulator truly changed 51 genes in spleen, 44 in kidney and 35
in liver. The top of the spleen list is dominated by named infection-response genes and by tissue-specific `g…` genes,
all truly changed — but further down the list, true and false hits mix. First look at all $p$-values at once.

In [ ]:
# ── 8.2 p-value histograms: a uniform floor of nulls plus a spike of real effects ──
fig, axes = plt.subplots(1, 3, figsize=(11, 3.1), sharey=True)
for ax, tissue in zip(axes, tissues):
    d = de[tissue]
    ax.hist([d.loc[~d.truly_changed, "p"], d.loc[d.truly_changed, "p"]], bins=20, range=(0, 1), stacked=True,
            color=["0.7", "tab:red"], label=["unchanged genes", "truly changed genes"], edgecolor="white")
    ax.axhline((~d.truly_changed).sum() / 20, color="k", ls=":", lw=1, label="uniform level of the nulls")
    ax.set_title(tissue); ax.set_xlabel("p-value")
axes[0].set_ylabel("number of genes"); axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** In each tissue the grey bars (genes that did not change) form a roughly flat floor at about
$349/20\approx17$ genes per bin — the uniform null distribution of [Module 25 · 🚦 Hypothesis testing: errors, power, z- and t-tests](25_hypothesis_testing.ipynb) — while the red bars (truly changed genes)
pile up in the first bin. This "uniform + spike" shape is the signature of a healthy $p$-value histogram in genomics; its
flat part tells you roughly how many nulls there are, which is the idea behind Storey's $q$-value method. (A histogram
that rises towards $p=1$ or has bumps in the middle signals a problem with the test's assumptions.)

In [ ]:
# ── 8.3 Volcano plots and the truth table for three decision rules ──
fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.6), sharey=True)
summary = []
for ax, tissue in zip(axes, tissues):
    d = de[tissue]
    hit = d["p_BH"] <= 0.05
    for mask, col, lab in [(~hit & ~d.truly_changed, "0.75", "not called, unchanged"),
                           (~hit & d.truly_changed, "tab:orange", "missed (truly changed)"),
                           (hit & d.truly_changed, "tab:red", "BH discovery, true"),
                           (hit & ~d.truly_changed, "k", "BH discovery, FALSE")]:
        ax.scatter(d.loc[mask, "log2FC"], -np.log10(d.loc[mask, "p"]), s=9, color=col, label=lab)
    ax.axhline(-np.log10(0.05), color="0.5", ls=":", lw=1)
    ax.axhline(-np.log10(0.05 / 400), color="tab:blue", ls="--", lw=1)
    ax.set_title(tissue); ax.set_xlabel("log2 fold change (infected − control)")
    for rule, rej in [("no correction", d["p"] <= 0.05), ("Bonferroni", d["p"] <= 0.05 / 400), ("BH q = 0.05", hit)]:
        tp, fp = (rej & d.truly_changed).sum(), (rej & ~d.truly_changed).sum()
        summary.append({"tissue": tissue, "rule": rule, "discoveries": rej.sum(), "true": tp, "false": fp,
                        "precision": tp / max(rej.sum(), 1), "recall": tp / d.truly_changed.sum()})
axes[0].set_ylabel("−log10 p"); axes[0].legend(fontsize=6, loc="upper left")
fig.suptitle("volcano plots (dotted: p = 0.05; dashed: Bonferroni 0.05/400)", fontsize=10)
plt.tight_layout(); plt.show()
print(pd.DataFrame(summary).round(3).to_string(index=False))

🔍 **What just happened?** A **volcano plot** shows each gene's effect size (log2 fold change, $x$) against its evidence
($-\log_{10}p$, $y$): genes that changed a lot *and* reliably sit high in the two upper corners. The table quantifies the
three rules against the truth. **No correction** finds 64 genes in spleen but 15 of them are false (precision 0.77); in
total about 43 false discoveries across the three tissues, close to the $0.05\times(349+356+365)\approx54$ expected.
**Bonferroni** makes no false discoveries but finds only 14, 5 and 8 genes (recall 0.11–0.27). **BH** finds 39, 34 and 25
genes with just 1, 0 and 1 false discoveries (precision 0.96–1.0) and recall around 0.7. The orange "missed" genes are
mostly those with small fold changes: with 4 replicates per group, the power to detect a 1-log2-unit change is limited —
a power problem, not a multiple-testing problem.

**Closing the loop with sets (Module 03).** Which genes respond in all three tissues, and which are tissue-specific? The
discovered sets can be compared with the true sets by Venn diagrams and the Jaccard index.

In [ ]:
# ── 8.4 Venn diagrams: discovered vs truly changed genes across tissues ──
found = {t: set(de[t].loc[de[t]["p_BH"] <= 0.05, "gene"]) for t in tissues}
true_sets = {t: set(expr["true_de"][t]) for t in tissues}
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
draw_venn(true_sets, ax=axes[0], title="truly changed genes (the simulation's truth)")
draw_venn(found, ax=axes[1], title="BH discoveries at q = 0.05")
plt.tight_layout(); plt.show()
for t in tissues:
    print(f"{t:6s}: |true| = {len(true_sets[t]):2d}, |found| = {len(found[t]):2d}, "
          f"Jaccard(found, true) = {jaccard(found[t], true_sets[t]):.3f}")
core_true = set.intersection(*true_sets.values())
core_found = set.intersection(*found.values())
print(f"changed in all three tissues: truth {len(core_true)} genes, found {len(core_found)}: {sorted(core_found)}")

🔍 **What just happened?** The left diagram is the truth: 12 genes (the named infection-response genes) change in all
three tissues, and each tissue has its own specific genes. The right diagram shows what BH recovered: the same *structure*
— a shared core and tissue-specific sets — but every region is smaller, because recall is about 0.7 per tissue and
recall compounds: a gene must be found in **all three** tissues to land in the centre, so only 8 of the 12 core genes are
recovered there. The Jaccard indices between found and true sets (0.67–0.77) summarise the agreement. (The two named genes in the
spleen ∩ liver region, *C3* and *g147*, are a core gene missed in kidney and a gene truly shared by spleen and liver.) *Lesson:* intersections
of discovery lists ("genes changed in every tissue") are especially sensitive to limited power — a gene absent from an
intersection may simply have been missed in one tissue.

### 🌉 Bridge — Differential expression in practice

Every RNA-seq differential-expression tool (DESeq2, edgeR, limma) ends with exactly this step: one $p$-value per gene,
then Benjamini–Hochberg; "padj < 0.05" in a results table *is* a BH-adjusted $p$-value. Real pipelines differ in the test
(negative-binomial count models with information shared across genes to stabilise variances when replicates are few),
but the multiple-testing logic is the one you just implemented. See
[BiologyToML Module 24 · GLMs for count data](../../../../ml/tutorial/BiologyToML/notebooks/24_glm_count_data.ipynb), where BH is applied to a simulated RNA-seq
experiment and the realised FDR is measured against the truth, as here.

---
## 9 · Gene-set enrichment: is the interferon pathway over-represented?

> 📍 **Where we are.** A list of 39 spleen genes is hard to interpret one by one. Biologists ask instead: *which known
> pathways are over-represented on the list?* That is a $2\times2$ table — and an exact test.

Of $N = 400$ genes, $K = 4$ belong to the interferon-response pathway; our BH list contains $n$ genes, $k$ of them from the
pathway. Under $H_0$ ("the list is a random draw of $n$ genes with respect to pathway membership"), the number of pathway
genes on the list, $X$, follows the **hypergeometric distribution**:

$$P(X = k) = \frac{\binom{K}{k}\binom{N-K}{n-k}}{\binom{N}{n}}, \qquad p\text{-value} = P(X\ge k_{\text{obs}}) = \sum_{k\ge k_{\text{obs}}}P(X=k).$$

*In words:* count the ways to choose $k$ of the $K$ pathway genes and $n-k$ of the other $N-K$ genes, and divide by all ways
to choose $n$ genes from $N$ (equally likely choices; binomial coefficients from Module 18). The one-sided $p$-value is the
chance that a random list of the same length would contain at least as many pathway genes. This is exactly the one-sided
Fisher's exact test of the $2\times2$ table (on list / not on list × in pathway / not in pathway).

In [ ]:
# ── 9.1 Hypergeometric enrichment test by hand, checked with SciPy ──
def enrichment_test(found_genes, gene_set, universe_size):
    """One-sided hypergeometric (Fisher) test: is gene_set over-represented in found_genes? Returns k, K, n, p."""
    K, n = len(gene_set), len(found_genes)
    k = len(set(found_genes) & set(gene_set))
    p = sum(comb(K, j) * comb(universe_size - K, n - j) for j in range(k, min(K, n) + 1)) / comb(universe_size, n)
    return k, K, n, p


N_genes = len(genes)
spleen_hits = found["spleen"]
gene_sets = dict(INFECTION_PATHWAYS)
gene_sets["housekeeping"] = HOUSEKEEPING_GENES
rows_enr = []
for name, members in gene_sets.items():
    k, K, n, p_hand = enrichment_test(spleen_hits, members, N_genes)
    p_scipy = stats.hypergeom.sf(k - 1, N_genes, K, n)
    p_fisher = stats.fisher_exact([[k, K - k], [n - k, N_genes - K - n + k]], alternative="greater").pvalue
    rows_enr.append({"gene set": name, "on list k": k, "set size K": K, "list size n": n,
                     "expected k": n * K / N_genes, "p (by hand)": p_hand, "p hypergeom": p_scipy, "p Fisher": p_fisher})
print(pd.DataFrame(rows_enr).round(5).to_string(index=False))
print("interferon genes on the spleen list:", sorted(spleen_hits & set(INFECTION_PATHWAYS["interferon response"])))

🔍 **What just happened?** The spleen list has 39 of 400 genes, so a random list would contain on average
$39\times4/400 = 0.39$ interferon genes. Ours contains **3** (*IRF1*, *MX1*, *STAT1*; *ISG15* was missed): the hypergeometric
$p$-value is **0.0032**, identical in our hand-written sum, `stats.hypergeom` and one-sided `fisher_exact`. The interferon
response is significantly enriched among the spleen discoveries — as it should be, since the simulation made those genes
respond. So are cytokine signalling (3 of 4, $p = 0.0032$) and the acute-phase genes (all 4, $p = 8\times10^{-5}$). The
housekeeping genes (expected 1.2, observed 0) are not enriched ($p = 1$). With many gene sets tested (Gene Ontology has
thousands), the enrichment $p$-values themselves need a multiple-testing correction — BH again.

⚠️ **Caveats.** Enrichment tests assume genes are exchangeable draws, but co-regulated genes are correlated; and the result
depends on the "universe" — testing against all 20,000 human genes instead of the ~12,000 expressed in the tissue inflates
significance. Always state the background you used.

### 📘 Lecture notes — enrichment as a $2\times2$ test

- **Question:** is gene set $S$ ($K$ genes) over-represented in a list of $n$ discoveries out of $N$ tested genes?
- **Null distribution:** hypergeometric, $P(X=k) = \binom Kk\binom{N-K}{n-k}/\binom Nn$; expected $nK/N$.
- **$p$-value:** $P(X\ge k_{\text{obs}})$ = one-sided Fisher's exact test of the $2\times2$ table.
- **Example:** 3 of 4 interferon genes among 39 spleen discoveries of 400 genes → $p = 0.0032$.
- **Practice:** correct across gene sets (BH); choose the background universe carefully; correlated genes violate the
  independence assumption.

> 🎓 **Exam tip.** Draw the 2 × 2 table (on list / not × in set / not) with all four counts before computing anything — the margins $n$, $K$, $N$ fix the hypergeometric distribution.

### 🌉 Bridge — Gene Ontology and gene-set enrichment analysis

The Gene Ontology (Ashburner *et al.* 2000) annotates genes with biological processes, molecular functions and cellular
components; "GO enrichment" of a discovery list is exactly the hypergeometric test above, run over thousands of terms
with FDR control. **Gene Set Enrichment Analysis** (GSEA, Subramanian *et al.* 2005) avoids the arbitrary cut-off of a
discovery list: it ranks *all* genes by their statistic and asks whether a set's members cluster near the top. Both are
standard steps after differential expression and after GWAS (gene-based and pathway-based association), and the graph
structure of pathways leads to network methods ([BiologyToML Module 28 · Graph neural networks](../../../../ml/tutorial/BiologyToML/notebooks/28_graph_neural_networks.ipynb)).

✏️ **Your turn.** Repeat the enrichment test for the kidney and liver lists, and for the union of all 12 infection genes
as one set. Then use the *unadjusted* list (all genes with $p\le0.05$) instead of the BH list: does the enrichment
$p$-value get larger or smaller, and why?

## 📌 Key takeaways

- **Categorical data are tested by comparing observed with expected counts:** $X^2 = \sum(O-E)^2/E$, with expected
  counts $E_i = n\dot p_i$ in the denominator; df = categories − 1 − (estimated parameters).
- **Mendel's dihybrid cross** fits 9:3:3:1 well: $X^2 = 0.47$, 3 df, $p = 0.925$ — and a very small $X^2$ can itself be
  suspicious (lower tail), as Fisher argued for Mendel's pooled data.
- **The χ² distribution** is the distribution of a sum of $p$ squared standard normals (mean $p$, variance $2p$); for two
  categories $X^2$ is exactly the squared standardised binomial count.
- **Hardy–Weinberg:** estimate $\hat p=(n_1+2n_2)/2n$, test on **1 df**; heterozygote deficits reveal inbreeding or
  genotyping errors.
- **Independence in contingency tables:** $E_{ij} = R_iC_j/n$, df $=(r-1)(c-1)$; allelic $2\times2$ tests and odds ratios
  drive case–control association studies; use **Fisher's exact test** for small counts.
- **Many tests:** $E[V] = m\alpha$ false positives; FWER $=1-(1-\alpha)^m$; **Bonferroni** ($p\le\alpha/m$) controls the FWER
  by the union bound but loses power.
- **Benjamini–Hochberg** (largest $k$ with $p_{(k)}\le kq/m$) controls the **false discovery rate** at
  $\frac{m_0}{m}q$; in our screens it kept ≈ 95 % of discoveries true with about 8× Bonferroni's power.
- **A genome-wide screen** = per-gene test + BH + evaluation; $p$-value histograms (uniform + spike), volcano plots, and
  **enrichment** tests (hypergeometric) turn gene lists into biology.

---
## ✅ Checkpoint

1. A cross predicts phenotypes in the ratio 1 : 2 : 1. Of 200 offspring, 40, 110 and 50 fall in the three classes.
   Compute $X^2$, its df and decide at α = 0.05.
   <details><summary>Answer</summary>Expected 50, 100, 50. $X^2 = \frac{(40-50)^2}{50}+\frac{(110-100)^2}{100}+\frac{(50-50)^2}{50} = 2+1+0 = 3.0$, df = 2. Cut-off 5.991 → retain $H_0$ ($p = e^{-3/2} = 0.223$; for 2 df the χ² tail is exactly $e^{-x/2}$).</details>
2. A classmate divides by $\dot p_i$ instead of $n\dot p_i$ for Mendel's data and gets 261. Explain the error and its
   consequence.
   <details><summary>Answer</summary>$\sum(O-E)^2/\dot p_i = n\sum(O-E)^2/E = 556\times0.47$. The statistic is inflated by the sample size, so it no longer has a χ² distribution under $H_0$; it would reject almost any theory in a large study.</details>
3. Why does a Hardy–Weinberg test have 1 degree of freedom although there are 3 genotype classes?
   <details><summary>Answer</summary>3 categories − 1 (counts sum to $n$) − 1 (the allele frequency $p$ is estimated from the same data) = 1.</details>
4. Show that for two categories $X^2 = Z^2$ with $Z = (O_1-n\dot p)/\sqrt{n\dot p(1-\dot p)}$.
   <details><summary>Answer</summary>$O_2-E_2 = -(O_1-E_1)$, so $X^2 = (O_1-E_1)^2(\frac1{n\dot p}+\frac1{n(1-\dot p)}) = (O_1-E_1)^2\frac{1}{n\dot p(1-\dot p)} = Z^2$.</details>
5. In a $2\times2$ table, 30 of 100 cases and 15 of 100 controls carry a variant. Compute the expected count of carrier
   cases under independence and the odds ratio.
   <details><summary>Answer</summary>Carriers: 45 of 200. $E = 100\times45/200 = 22.5$. Odds ratio $=\frac{30/70}{15/85} = \frac{0.4286}{0.1765}\approx2.43$.</details>
6. You test 50 genes at α = 0.05; all nulls are true and independent. What are $E[V]$ and the FWER? What Bonferroni
   threshold keeps the FWER ≤ 0.05?
   <details><summary>Answer</summary>$E[V] = 2.5$; FWER $=1-0.95^{50}\approx0.923$; threshold $0.05/50 = 0.001$.</details>
7. Prove that Bonferroni controls the FWER without assuming independence.
   <details><summary>Answer</summary>FWER $=P(\bigcup_{i\in\mathcal H_0}A_i)\le\sum_{i\in\mathcal H_0}P(A_i)=m_0\alpha/m\le\alpha$ by the union bound $P(\bigcup A_i)\le\sum P(A_i)$, which holds for any events.</details>
8. Run BH at $q = 0.1$ on $p$ = 0.002, 0.03, 0.035, 0.04, 0.2, 0.5 ($m = 6$).
   <details><summary>Answer</summary>Thresholds $k\cdot0.1/6$: 0.0167, 0.0333, 0.05, 0.0667, 0.0833, 0.1. Passes: 0.002 ✓, 0.03 ✓, 0.035 ✓, 0.04 ✓, 0.2 ✗, 0.5 ✗ → largest $k=4$: reject the four smallest.</details>
9. What is the difference between controlling the FWER and the FDR? Give a biological setting for each.
   <details><summary>Answer</summary>FWER bounds the probability of even one false positive — appropriate when each claim is costly (a confirmatory clinical trial with several endpoints; declaring a GWAS hit). FDR bounds the expected fraction of false positives among discoveries — appropriate for exploratory screens (a DE gene list followed up by validation experiments).</details>
10. A $p$-value histogram from a screen of 10,000 genes is flat at about 450 genes per 0.05-bin, with 1,300 genes in the
    first bin. Roughly how many genes are truly null, and how many discoveries at $p\le0.05$ are false?
    <details><summary>Answer</summary>Flat level 450 per bin × 20 bins ≈ 9,000 nulls; of the 1,300 genes with $p\le0.05$, about 450 are nulls (false) and ≈ 850 real — a false discovery proportion ≈ 0.35.</details>
11. A pathway has 10 of 1,000 genes; your list has 50 genes, 4 from the pathway. What is the expected number under $H_0$,
    and which distribution gives the $p$-value?
    <details><summary>Answer</summary>Expected $50\times10/1000 = 0.5$. $X\sim$ Hypergeometric($N=1000$, $K=10$, $n=50$); $p = P(X\ge4)$ = `stats.hypergeom.sf(3, 1000, 10, 50)` ≈ 0.0009.</details>

## 🧠 Questions & discussion

1. Fisher's analysis suggested Mendel's data fit "too well". What would you need, as a historian of science, to
   distinguish deliberate adjustment, unconscious bias and selective reporting? Why does none of them affect the truth of
   Mendel's laws?
2. A GWAS reports a SNP with $p = 3\times10^{-7}$. Is it a "hit"? Discuss genome-wide significance, replication in an
   independent cohort and the role of sample size.
3. Hardy–Weinberg tests are usually applied to **controls** only. Why might cases deviate from equilibrium at a truly
   associated SNP?
4. BH assumes independent (or positively dependent) tests. Genes in a pathway are correlated. What could go wrong, and how
   could you check with simulation?
5. Two labs analyse the same RNA-seq data; one reports 500 DE genes at FDR 5 %, the other 50 at FWER 5 %. Are they
   contradicting each other?

## 🚀 Bonus — going further

- **Storey's $q$-value.** Estimate the fraction of true nulls $\hat\pi_0$ from the flat part of the $p$-value histogram
  (e.g. the density above $p = 0.5$) and use the less conservative threshold $kq/(m\hat\pi_0)$ (Storey & Tibshirani 2003).
- **An exact Hardy–Weinberg test.** For small samples, enumerate all heterozygote counts compatible with the observed
  allele counts and compute their exact probabilities (Wigginton *et al.* 2005); compare with the χ² test.
- **The likelihood-ratio $G$-test.** Compute $G = 2\sum O_i\log(O_i/E_i)$ for Mendel's data and compare with $X^2$; both
  are asymptotically $\chi^2_{k-1}$.
- **Permutation FDR.** For the spleen data, shuffle the condition labels many times, record how many genes pass a $p$
  threshold in each shuffle, and estimate the FDR of that threshold empirically.
- **Holm's step-down procedure** controls the FWER and is uniformly more powerful than Bonferroni — implement it and
  compare on the screen of Section 6.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 26 — Categorical data and multiple testing            Date:
Input: Mendel's dihybrid counts (typed in); SIMULATED genotypes (simulate_genotypes: HWE, inbreeding, case–control);
       SIMULATED screens of 10,000 z-tests; SIMULATED infection experiment (400 genes × 3 tissues, 4 vs 4)
Output: X² statistics and p-values; FWER/FDR of Bonferroni and BH; DE gene lists, precision/recall; enrichment p-values
Method: chi-squared goodness of fit / independence; Fisher's exact (hypergeometric); Bonferroni; Benjamini–Hochberg
Key equation: X² = Σ (O − E)²/E ~ χ²_df;  E_ij = R_i C_j / n;  BH: largest k with p_(k) ≤ k q / m
Key parameters: degrees of freedom; alpha; q; number of tests m; replicates per group
What changed when I modified …: (e.g. inbreeding F, sample size, effect shift, q)
One failure mode: (e.g. dividing by ṗ instead of n·ṗ; forgetting the df for estimated parameters; no multiple-testing correction)
Connection to 02-680: χ² tests for categorical data; multiple testing (FWER, FDR)
```

## 📚 References

- Wasserman L. (2004) *All of Statistics*. Springer — Chapter 10 (χ² tests, multiple testing).
- DeGroot M. H., Schervish M. J. (2012) *Probability and Statistics*, 4th ed. Pearson — Chapters 9 and 10 (χ² tests,
  contingency tables).
- Betti M. *Mathematics and Statistics for the Quantitative Sciences*. CRC Press.
- D. DeBlasio, 02-680 lecture notes, github.com/deblasiolab/02-680_Documents.
- Mendel G. (1866) Versuche über Pflanzen-Hybriden. *Verhandlungen des naturforschenden Vereines in Brünn* 4:3–47.
- Pearson K. (1900) On the criterion that a given system of deviations from the probable in the case of a correlated
  system of variables is such that it can be reasonably supposed to have arisen from random sampling.
  *Philosophical Magazine* 50:157–175.
- Fisher R. A. (1936) Has Mendel's work been rediscovered? *Annals of Science* 1:115–137.
- Franklin A., Edwards A. W. F., Fairbanks D. J., Hartl D. L., Seidenfeld T. (2008) *Ending the Mendel–Fisher
  Controversy*. University of Pittsburgh Press.
- Benjamini Y., Hochberg Y. (1995) Controlling the false discovery rate: a practical and powerful approach to multiple
  testing. *Journal of the Royal Statistical Society B* 57:289–300.
- Storey J. D., Tibshirani R. (2003) Statistical significance for genomewide studies. *PNAS* 100:9440–9445.
- Wigginton J. E., Cutler D. J., Abecasis G. R. (2005) A note on exact tests of Hardy–Weinberg equilibrium.
  *American Journal of Human Genetics* 76:887–893.
- Ashburner M. *et al.* (2000) Gene Ontology: tool for the unification of biology. *Nature Genetics* 25:25–29.
- Subramanian A. *et al.* (2005) Gene set enrichment analysis: a knowledge-based approach for interpreting genome-wide
  expression profiles. *PNAS* 102:15545–15550.

## ➡️ Next

[Module 27 · 📉 Linear regression: the whole course in one model](27_linear_regression_review.ipynb) ties the whole course together in one model. Linear regression uses vectors, matrices, rank and projections for its
geometry, calculus for its least-squares solution, probability and maximum likelihood for its noise model, MAP for ridge
regression — and the tests of Modules 25–26 for its coefficients: a $t$-test per coefficient, and a regression on a 0/1
predictor that *is* the two-sample $t$-test. A GWAS is then simply one regression per SNP followed by the multiple-testing
correction of this module.